In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ======================================================
# 1. Load data
# ======================================================
data = pd.read_csv("SKCM_transposed.csv")

X_full = data.drop(columns=["sample", "index", "TumorType"], errors="ignore")
y_full = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X_full.shape)
print("Class distribution:", np.bincount(y_full))

# ======================================================
# 2. 5-fold CV with feature selection inside each fold
#    - RF on training fold -> top10 & top20
#    - LR on top10
#    - RF on top20
#    - Soft voting (average of probas)
# ======================================================

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics_lr  = {"acc": [], "prec": [], "rec": [], "f1": []}
metrics_rf  = {"acc": [], "prec": [], "rec": [], "f1": []}
metrics_ens = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full, y_full), start=1):
    print(f"\n=== Fold {fold}/{n_splits} ===")

    X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]

    # ---------------- RF for feature selection (on training set only) ----------------
    rf_fs = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )
    rf_fs.fit(X_train, y_train)

    importances = rf_fs.feature_importances_
    feature_names = np.array(X_full.columns)

    idx_sorted = np.argsort(importances)[::-1]
    feature_sorted = feature_names[idx_sorted]

    top10 = feature_sorted[:10]
    top20 = feature_sorted[:20]

    # Subsets
    X_train_top10 = X_train[top10]
    X_test_top10  = X_test[top10]

    X_train_top20 = X_train[top20]
    X_test_top20  = X_test[top20]

    # ---------------- Logistic Regression on top 10 ----------------
    lr = LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )
    lr.fit(X_train_top10, y_train)
    y_pred_lr = lr.predict(X_test_top10)

    acc_lr  = accuracy_score(y_test, y_pred_lr)
    prec_lr = precision_score(y_test, y_pred_lr, zero_division=0)
    rec_lr  = recall_score(y_test, y_pred_lr, zero_division=0)
    f1_lr   = f1_score(y_test, y_pred_lr, zero_division=0)

    metrics_lr["acc"].append(acc_lr)
    metrics_lr["prec"].append(prec_lr)
    metrics_lr["rec"].append(rec_lr)
    metrics_lr["f1"].append(f1_lr)

    print(f"  LR (top10)  -> Acc={acc_lr:.4f}, Prec={prec_lr:.4f}, Rec={rec_lr:.4f}, F1={f1_lr:.4f}")

    # ---------------- RF on top 20 ----------------
    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_top20, y_train)
    y_pred_rf = rf.predict(X_test_top20)

    acc_rf  = accuracy_score(y_test, y_pred_rf)
    prec_rf = precision_score(y_test, y_pred_rf, zero_division=0)
    rec_rf  = recall_score(y_test, y_pred_rf, zero_division=0)
    f1_rf   = f1_score(y_test, y_pred_rf, zero_division=0)

    metrics_rf["acc"].append(acc_rf)
    metrics_rf["prec"].append(prec_rf)
    metrics_rf["rec"].append(rec_rf)
    metrics_rf["f1"].append(f1_rf)

    print(f"  RF (top20)  -> Acc={acc_rf:.4f}, Prec={prec_rf:.4f}, Rec={rec_rf:.4f}, F1={f1_rf:.4f}")

    # ---------------- Soft voting ensemble ----------------
    proba_lr = lr.predict_proba(X_test_top10)   # (n, 2)
    proba_rf = rf.predict_proba(X_test_top20)   # (n, 2)
    proba_ens = (proba_lr + proba_rf) / 2.0

    y_pred_ens = (proba_ens[:, 1] >= 0.5).astype(int)

    acc_ens  = accuracy_score(y_test, y_pred_ens)
    prec_ens = precision_score(y_test, y_pred_ens, zero_division=0)
    rec_ens  = recall_score(y_test, y_pred_ens, zero_division=0)
    f1_ens   = f1_score(y_test, y_pred_ens, zero_division=0)

    metrics_ens["acc"].append(acc_ens)
    metrics_ens["prec"].append(prec_ens)
    metrics_ens["rec"].append(rec_ens)
    metrics_ens["f1"].append(f1_ens)

    print(f"  Ensemble    -> Acc={acc_ens:.4f}, Prec={prec_ens:.4f}, Rec={rec_ens:.4f}, F1={f1_ens:.4f}")

# ======================================================
# 3. Summaries: mean ± std over 5 folds
# ======================================================

def summarize(name, m):
    print(f"\n{name} performance over {n_splits} folds:")
    print(f"  Accuracy : {np.mean(m['acc']):.4f} ± {np.std(m['acc']):.4f}")
    print(f"  Precision: {np.mean(m['prec']):.4f} ± {np.std(m['prec']):.4f}")
    print(f"  Recall   : {np.mean(m['rec']):.4f} ± {np.std(m['rec']):.4f}")
    print(f"  F1-score : {np.mean(m['f1']):.4f} ± {np.std(m['f1']):.4f}")

summarize("Logistic Regression (top10)", metrics_lr)
summarize("Random Forest (top20)", metrics_rf)
summarize("Soft Voting Ensemble", metrics_ens)

X shape: (473, 20530)
Class distribution: [104 369]

=== Fold 1/5 ===
  LR (top10)  -> Acc=0.8947, Prec=0.9103, Rec=0.9595, F1=0.9342
  RF (top20)  -> Acc=0.8737, Prec=0.9079, Rec=0.9324, F1=0.9200
  Ensemble    -> Acc=0.8842, Prec=0.8987, Rec=0.9595, F1=0.9281

=== Fold 2/5 ===
  LR (top10)  -> Acc=0.8737, Prec=0.8875, Rec=0.9595, F1=0.9221
  RF (top20)  -> Acc=0.8842, Prec=0.8987, Rec=0.9595, F1=0.9281
  Ensemble    -> Acc=0.8947, Prec=0.9000, Rec=0.9730, F1=0.9351

=== Fold 3/5 ===
  LR (top10)  -> Acc=0.9053, Prec=0.9114, Rec=0.9730, F1=0.9412
  RF (top20)  -> Acc=0.8737, Prec=0.8974, Rec=0.9459, F1=0.9211
  Ensemble    -> Acc=0.8947, Prec=0.9000, Rec=0.9730, F1=0.9351

=== Fold 4/5 ===
  LR (top10)  -> Acc=0.8723, Prec=0.8974, Rec=0.9459, F1=0.9211
  RF (top20)  -> Acc=0.8830, Prec=0.9200, Rec=0.9324, F1=0.9262
  Ensemble    -> Acc=0.8936, Prec=0.9103, Rec=0.9595, F1=0.9342

=== Fold 5/5 ===
  LR (top10)  -> Acc=0.9043, Prec=0.8902, Rec=1.0000, F1=0.9419
  RF (top20)  -> Acc=0.904

In [ ]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE  # from imbalanced-learn

# ======================================================
# 1. Load data
# ======================================================
data = pd.read_csv("SKCM_transposed.csv")

X_full = data.drop(columns=["sample", "index", "TumorType"], errors="ignore")
y_full = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X_full.shape)
print("Class distribution:", np.bincount(y_full))

# ======================================================
# 2. CV + SMOTE + PCA + AdaBoost
# ======================================================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# choose how many principal components you want to keep
n_components = 20  # you can adjust (e.g., 10, 20, 50)

metrics_ab = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full, y_full), start=1):
    print(f"\n=== Fold {fold}/{n_splits} ===")

    X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]

    # ---------- SMOTE oversampling (k=3) on training data ----------
    sm = SMOTE(k_neighbors=3, random_state=42)
    X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
    print("  After SMOTE - class distribution:", np.bincount(y_train_sm))

    # ---------- PCA (fit on *resampled* train, transform both train and test) ----------
    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_sm)
    X_test_pca = pca.transform(X_test)

    # ---------- AdaBoost (DT base, depth=6, 12 trees) ----------
    base_dt = DecisionTreeClassifier(max_depth=6, random_state=42)
    ab_clf = AdaBoostClassifier(
        n_estimators=12,
        learning_rate=1.0,
        random_state=42
    )

    ab_clf.fit(X_train_pca, y_train_sm)
    y_pred = ab_clf.predict(X_test_pca)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
    rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

    metrics_ab["acc"].append(acc)
    metrics_ab["prec"].append(prec)
    metrics_ab["rec"].append(rec)
    metrics_ab["f1"].append(f1)

    print(f"  AdaBoost+SMOTE+PCA -> Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

# ======================================================
# 3. Summaries: mean ± std over folds
# ======================================================
print("\nAdaBoost (DT depth=6, 12 trees) with SMOTE(k=3) + PCA performance:")
print(f"  Accuracy : {np.mean(metrics_ab['acc']):.4f} ± {np.std(metrics_ab['acc']):.4f}")
print(f"  Precision: {np.mean(metrics_ab['prec']):.4f} ± {np.std(metrics_ab['prec']):.4f}")
print(f"  Recall   : {np.mean(metrics_ab['rec']):.4f} ± {np.std(metrics_ab['rec']):.4f}")
print(f"  F1-score : {np.mean(metrics_ab['f1']):.4f} ± {np.std(metrics_ab['f1']):.4f}")

X shape: (473, 20530)
Class distribution: [104 369]

=== Fold 1/5 ===
  After SMOTE - class distribution: [295 295]
  AdaBoost+SMOTE+PCA -> Acc=0.7579, Prec=0.9322, Rec=0.7432, F1=0.8271

=== Fold 2/5 ===
  After SMOTE - class distribution: [295 295]
  AdaBoost+SMOTE+PCA -> Acc=0.7789, Prec=0.8955, Rec=0.8108, F1=0.8511

=== Fold 3/5 ===
  After SMOTE - class distribution: [295 295]
  AdaBoost+SMOTE+PCA -> Acc=0.8105, Prec=0.9000, Rec=0.8514, F1=0.8750

=== Fold 4/5 ===
  After SMOTE - class distribution: [295 295]
  AdaBoost+SMOTE+PCA -> Acc=0.8085, Prec=0.9000, Rec=0.8514, F1=0.8750

=== Fold 5/5 ===
  After SMOTE - class distribution: [296 296]
  AdaBoost+SMOTE+PCA -> Acc=0.7447, Prec=0.9153, Rec=0.7397, F1=0.8182

AdaBoost (DT depth=6, 12 trees) with SMOTE(k=3) + PCA performance:
  Accuracy : 0.7801 ± 0.0264
  Precision: 0.9086 ± 0.0136
  Recall   : 0.7993 ± 0.0495
  F1-score : 0.8493 ± 0.0236


In [ ]:
import numpy as np
import pandas as pd

from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ======================================================
# 1. Load data
# ======================================================
data = pd.read_csv("SKCM_transposed.csv")

X_full = data.drop(columns=["sample", "index", "TumorType"], errors="ignore")
y_full = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X_full.shape)
print("Class distribution:", np.bincount(y_full))

# ======================================================
# 2. 5-fold CV: SVC-L1 feature selection (17 feats) + SVC training
# ======================================================

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics_svc = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full, y_full), start=1):
    print(f"\n=== Fold {fold}/{n_splits} ===")

    X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]

    # ---------- SVC-L1 for feature selection ----------
    # LinearSVC with L1 penalty gives sparse weights; we use them to rank features.
    l1_svc = LinearSVC(
        penalty="l1",
        dual=False,        # required for L1
        C=1.0,
        max_iter=5000,
        random_state=42
    )
    l1_svc.fit(X_train, y_train)

    # Absolute weights per feature (for binary, coef_ shape is (1, n_features))
    coef = np.abs(l1_svc.coef_[0])
    feature_names = np.array(X_full.columns)

    # Rank features by absolute weight
    idx_sorted = np.argsort(coef)[::-1]
    # Take top 17 features
    top17_idx = idx_sorted[:17]
    top17_features = feature_names[top17_idx]

    print("  Selected 17 features (by |w| from SVC-L1):")
    print("  ", ", ".join(top17_features))

    # Restrict train/test to these 17 features
    X_train_sel = X_train[top17_features]
    X_test_sel = X_test[top17_features]

    # ---------- Train final SVC on selected features ----------
    # You can use linear or RBF kernel; here we use RBF as a typical choice.
    svc = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=False,
        random_state=42
    )
    svc.fit(X_train_sel, y_train)

    y_pred = svc.predict(X_test_sel)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    metrics_svc["acc"].append(acc)
    metrics_svc["prec"].append(prec)
    metrics_svc["rec"].append(rec)
    metrics_svc["f1"].append(f1)

    print(f"  SVC (17 feats) -> Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

# ======================================================
# 3. Summaries: mean ± std over folds
# ======================================================

print("\nSVC with SVC-L1 feature selection (17 features) performance:")
print(f"  Accuracy : {np.mean(metrics_svc['acc']):.4f} ± {np.std(metrics_svc['acc']):.4f}")
print(f"  Precision: {np.mean(metrics_svc['prec']):.4f} ± {np.std(metrics_svc['prec']):.4f}")
print(f"  Recall   : {np.mean(metrics_svc['rec']):.4f} ± {np.std(metrics_svc['rec']):.4f}")
print(f"  F1-score : {np.mean(metrics_svc['f1']):.4f} ± {np.std(metrics_svc['f1']):.4f}")

X shape: (473, 20530)
Class distribution: [104 369]

=== Fold 1/5 ===


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  Selected 17 features (by |w| from SVC-L1):
   C7, MMP9, MMP3, CILP, LRRC15, F2RL2, KCND2, KDELC1, DDX43, NRG3, KRT17, CHRDL2, FCER1A, TFAP2B, SH3GL2, ZNF883, ?|652919
  SVC (17 feats) -> Acc=0.9263, Prec=0.9467, Rec=0.9595, F1=0.9530

=== Fold 2/5 ===


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  Selected 17 features (by |w| from SVC-L1):
   F2RL2, EBF2, STMN2, C7, VSTM2B, HORMAD1, NEFL, KRT17, CTNND2, NRAP, MAGEA1, VWDE, MMP3, DPYSL5, RSPO4, SORCS2, LOC645323
  SVC (17 feats) -> Acc=0.9263, Prec=0.9351, Rec=0.9730, F1=0.9536

=== Fold 3/5 ===


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  Selected 17 features (by |w| from SVC-L1):
   C7, EBF2, LRRC15, LOC220594, STMN2, CD5L, MME, FAM182B, RGPD8, FAM70A, KRT4, DNAH6, NEFL, F2RL2, FOSB, MMP3, ZNF883
  SVC (17 feats) -> Acc=0.9263, Prec=0.9241, Rec=0.9865, F1=0.9542

=== Fold 4/5 ===


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  Selected 17 features (by |w| from SVC-L1):
   C7, MMP3, STMN2, ITIH4, KRT17, MASP1, HBA2, CACNA2D2, MMP9, PRSS50, MMP1, DLL3, PIWIL1, PRIMA1, DPP6, CSAG3, LOC440925
  SVC (17 feats) -> Acc=0.9468, Prec=0.9859, Rec=0.9459, F1=0.9655

=== Fold 5/5 ===
  Selected 17 features (by |w| from SVC-L1):
   EBF2, C7, STMN2, FABP4, KCNH2, MMP3, DNASE2B, MCTP1, PRSS50, DDX43, KRT17, MMP9, MYOZ1, WDR63, NEFL, STAP2, NRK
  SVC (17 feats) -> Acc=0.9362, Prec=0.9351, Rec=0.9863, F1=0.9600

SVC with SVC-L1 feature selection (17 features) performance:
  Accuracy : 0.9324 ± 0.0082
  Precision: 0.9454 ± 0.0215
  Recall   : 0.9702 ± 0.0157
  F1-score : 0.9573 ± 0.0048


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ======================================================
# 1. Load data
# ======================================================
data = pd.read_csv("SKCM_transposed.csv")

X = data.drop(columns=["sample", "index", "TumorType"], errors="ignore")
y = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# ======================================================
# 2. 5-fold CV: Random Forest on all features
# ======================================================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics_rf = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold}/{n_splits} ===")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    metrics_rf["acc"].append(acc)
    metrics_rf["prec"].append(prec)
    metrics_rf["rec"].append(rec)
    metrics_rf["f1"].append(f1)

    print(f"  RF (all features) -> Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

# ======================================================
# 3. Summaries: mean ± std over folds
# ======================================================
print("\nRandom Forest on whole feature set performance:")
print(f"  Accuracy : {np.mean(metrics_rf['acc']):.4f} ± {np.std(metrics_rf['acc']):.4f}")
print(f"  Precision: {np.mean(metrics_rf['prec']):.4f} ± {np.std(metrics_rf['prec']):.4f}")
print(f"  Recall   : {np.mean(metrics_rf['rec']):.4f} ± {np.std(metrics_rf['rec']):.4f}")
print(f"  F1-score : {np.mean(metrics_rf['f1']):.4f} ± {np.std(metrics_rf['f1']):.4f}")

X shape: (473, 20530)
Class distribution: [104 369]

=== Fold 1/5 ===
  RF (all features) -> Acc=0.8737, Prec=0.8875, Rec=0.9595, F1=0.9221

=== Fold 2/5 ===
  RF (all features) -> Acc=0.8842, Prec=0.8795, Rec=0.9865, F1=0.9299

=== Fold 3/5 ===
  RF (all features) -> Acc=0.8632, Prec=0.8765, Rec=0.9595, F1=0.9161

=== Fold 4/5 ===
  RF (all features) -> Acc=0.8511, Prec=0.8488, Rec=0.9865, F1=0.9125

=== Fold 5/5 ===
  RF (all features) -> Acc=0.8830, Prec=0.8780, Rec=0.9863, F1=0.9290

Random Forest on whole feature set performance:
  Accuracy : 0.8710 ± 0.0125
  Precision: 0.8741 ± 0.0132
  Recall   : 0.9756 ± 0.0132
  F1-score : 0.9219 ± 0.0069


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# ======================================================
# 1. Load data
# ======================================================
data = pd.read_csv("SKCM_transposed.csv")

# List of genes to keep
selected_genes = [
    "TTYH3", "NME1", "ORC1", "PLK1", "MYO10", "SPINT1", "NUPR1", "SERPINE2",
    "HLA-DQB2", "METTL7B", "TIMP1", "NOX4", "DBI", "ARL15", "APOBEC3G",
    "ARRB2", "DRAM1", "RNF213", "C14orf28", "CPEB3"
]

# Keep only genes that actually exist in the file
available_genes = [g for g in selected_genes if g in data.columns]
if len(available_genes) == 0:
    raise ValueError("None of the selected genes are present in SKCM_transposed.csv.")

missing_genes = set(selected_genes) - set(available_genes)
if missing_genes:
    print("Warning: these selected genes are not in the dataset and will be ignored:")
    print(", ".join(sorted(missing_genes)))

print(f"\nUsing {len(available_genes)} genes:", ", ".join(available_genes))

# Features and labels
X = data[available_genes].copy()
y = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# ======================================================
# 2. 5-fold CV with Extra Trees
# ======================================================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics_et = {"acc": [], "prec": [], "rec": [], "f1": []}
cms = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold}/{n_splits} ===")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    et = ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    )
    et.fit(X_train, y_train)
    y_pred = et.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    metrics_et["acc"].append(acc)
    metrics_et["prec"].append(prec)
    metrics_et["rec"].append(rec)
    metrics_et["f1"].append(f1)
    cms.append(cm)

    print(f"  ExtraTrees -> Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")
    print("  Confusion matrix:\n", cm)

# ======================================================
# 3. Summary: mean ± std and combined confusion matrix
# ======================================================
print("\nExtraTrees on selected genes performance (5-fold CV):")
print(f"  Accuracy : {np.mean(metrics_et['acc']):.4f} ± {np.std(metrics_et['acc']):.4f}")
print(f"  Precision: {np.mean(metrics_et['prec']):.4f} ± {np.std(metrics_et['prec']):.4f}")
print(f"  Recall   : {np.mean(metrics_et['rec']):.4f} ± {np.std(metrics_et['rec']):.4f}")
print(f"  F1-score : {np.mean(metrics_et['f1']):.4f} ± {np.std(metrics_et['f1']):.4f}")

cm_sum = np.sum(cms, axis=0)
print("\nSummed confusion matrix over 5 folds:")
print(cm_sum)

print("\nClassification report on all folds combined (approximated using summed CM):")
tp = cm_sum[1, 1]
tn = cm_sum[0, 0]
fp = cm_sum[0, 1]
fn = cm_sum[1, 0]
precision_overall = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_overall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_overall = (
    2 * precision_overall * recall_overall / (precision_overall + recall_overall)
    if (precision_overall + recall_overall) > 0 else 0.0
)
acc_overall = (tp + tn) / cm_sum.sum() if cm_sum.sum() > 0 else 0.0
print(f"  Accuracy : {acc_overall:.4f}")
print(f"  Precision: {precision_overall:.4f}")
print(f"  Recall   : {recall_overall:.4f}")
print(f"  F1-score : {f1_overall:.4f}")

ORC1

Using 19 genes: TTYH3, NME1, PLK1, MYO10, SPINT1, NUPR1, SERPINE2, HLA-DQB2, METTL7B, TIMP1, NOX4, DBI, ARL15, APOBEC3G, ARRB2, DRAM1, RNF213, C14orf28, CPEB3
X shape: (473, 19)
Class distribution: [104 369]

=== Fold 1/5 ===
  ExtraTrees -> Acc=0.7895, Prec=0.7872, Rec=1.0000, F1=0.8810
  Confusion matrix:
 [[ 1 20]
 [ 0 74]]

=== Fold 2/5 ===
  ExtraTrees -> Acc=0.8000, Prec=0.8090, Rec=0.9730, F1=0.8834
  Confusion matrix:
 [[ 4 17]
 [ 2 72]]

=== Fold 3/5 ===
  ExtraTrees -> Acc=0.8000, Prec=0.7957, Rec=1.0000, F1=0.8862
  Confusion matrix:
 [[ 2 19]
 [ 0 74]]

=== Fold 4/5 ===
  ExtraTrees -> Acc=0.8085, Prec=0.8043, Rec=1.0000, F1=0.8916
  Confusion matrix:
 [[ 2 18]
 [ 0 74]]

=== Fold 5/5 ===
  ExtraTrees -> Acc=0.8298, Prec=0.8202, Rec=1.0000, F1=0.9012
  Confusion matrix:
 [[ 5 16]
 [ 0 73]]

ExtraTrees on selected genes performance (5-fold CV):
  Accuracy : 0.8056 ± 0.0135
  Precision: 0.8033 ± 0.0113
  Recall   : 0.9946 ± 0.0108
  F1-score : 0.8887 ± 0.0072

Summed co

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =====================================================
# 1. Load data
# =====================================================
data = pd.read_csv("SKCM_ANOVA_500.csv")

# Features: all except TumorType
X = data.drop(columns=["TumorType"], errors="ignore")
# Labels: binary 0/1 (Metastatic vs others)
y = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =====================================================
# 2. Define MLP and hyperparameter grid
# =====================================================
mlp = MLPClassifier(max_iter=400, random_state=42)

# Start with a *moderate* grid first. You can expand later if runtime is OK.
param_grid = {
    "hidden_layer_sizes": [
        (64,),         # 1 layer
        (128, 64),     # 2 layers
        (256, 128),    # deeper
    ],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [1e-4, 1e-3],          # L2 regularization
    "learning_rate_init": [1e-3, 5e-4],
    "batch_size": [32, 64]
}

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring="accuracy",
    n_jobs=-1,
    cv=cv_inner,
    verbose=1
)

print("\nRunning GridSearchCV for MLP hyperparameters...\n")
grid.fit(X, y)

print("\nBest parameters from GridSearchCV:")
print(grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.4f}")

best_params = grid.best_params_

# =====================================================
# 3. Evaluate MLP with best hyperparameters (outer CV)
# =====================================================
print("\nEvaluating tuned MLP with 5-fold cross-validation...\n")

mlp_best = MLPClassifier(
    **best_params,
    max_iter=400,
    random_state=42
)

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get out-of-fold predictions for metrics
y_pred_oof = cross_val_predict(
    mlp_best,
    X,
    y,
    cv=cv_outer,
    n_jobs=-1
)

acc = accuracy_score(y, y_pred_oof)
prec = precision_score(y, y_pred_oof, average="binary", zero_division=0)
rec = recall_score(y, y_pred_oof, average="binary", zero_division=0)
f1 = f1_score(y, y_pred_oof, average="binary", zero_division=0)
cm = confusion_matrix(y, y_pred_oof)

print("Overall performance with tuned MLP (5-fold OOF):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")
print("\nConfusion matrix:")
print(cm)
print("\nClassification report:")
print(classification_report(y, y_pred_oof, digits=4))

X shape: (473, 500)
Class distribution: [104 369]

Running GridSearchCV for MLP hyperparameters...

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters from GridSearchCV:
{'activation': 'relu', 'alpha': 0.0001, 'batch_size': 64, 'hidden_layer_sizes': (256, 128), 'learning_rate_init': 0.0005, 'solver': 'adam'}
Best CV accuracy: 0.9366

Evaluating tuned MLP with 5-fold cross-validation...

Overall performance with tuned MLP (5-fold OOF):
  Accuracy : 0.9366
  Precision: 0.9449
  Recall   : 0.9756
  F1-score : 0.9600

Confusion matrix:
[[ 83  21]
 [  9 360]]

Classification report:
              precision    recall  f1-score   support

           0     0.9022    0.7981    0.8469       104
           1     0.9449    0.9756    0.9600       369

    accuracy                         0.9366       473
   macro avg     0.9235    0.8868    0.9035       473
weighted avg     0.9355    0.9366    0.9351       473



In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =====================================================
# 1. Load data
# =====================================================
data = pd.read_csv("Final_RF.csv")

# Features: all except TumorType
X = data.drop(columns=["TumorType"], errors="ignore")
# Labels: binary 0/1 (Metastatic vs others)
y = (data["TumorType"])

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =====================================================
# 2. Fixed MLP with your best hyperparameters
# =====================================================
best_params = {
    'activation': 'relu',
    'alpha': 0.0001,
    'batch_size': 64,
    'hidden_layer_sizes': (256, 128),
    'learning_rate_init': 0.0005,
    'solver': 'adam'
}

mlp_best = MLPClassifier(
    **best_params,
    max_iter=400,
    random_state=42
)

# =====================================================
# 3. 5-fold CV evaluation with these parameters
# =====================================================
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions to compute global metrics
y_pred_oof = cross_val_predict(
    mlp_best,
    X,
    y,
    cv=cv_outer,
    n_jobs=-1
)

acc = accuracy_score(y, y_pred_oof)
prec = precision_score(y, y_pred_oof, average="binary", zero_division=0)
rec = recall_score(y, y_pred_oof, average="binary", zero_division=0)
f1 = f1_score(y, y_pred_oof, average="binary", zero_division=0)
cm = confusion_matrix(y, y_pred_oof)

print("\nTuned MLP performance (5-fold OOF using fixed best params):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")
print("\nConfusion matrix:")
print(cm)
print("\nClassification report:")
print(classification_report(y, y_pred_oof, digits=4))

# Also show fold-wise stats if you want mean ± std
fold_accs, fold_precs, fold_recs, fold_f1s = [], [], [], []

for fold_id, (train_idx, test_idx) in enumerate(cv_outer.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf = MLPClassifier(
        **best_params,
        max_iter=400,
        random_state=42 + fold_id  # slightly vary seed across folds
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    fold_accs.append(accuracy_score(y_test, y_pred))
    fold_precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
    fold_recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
    fold_f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

print("\nFold-wise mean ± std with fixed tuned MLP:")
print(f"  Accuracy : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
print(f"  Precision: {np.mean(fold_precs):.4f} ± {np.std(fold_precs):.4f}")
print(f"  Recall   : {np.mean(fold_recs):.4f} ± {np.std(fold_recs):.4f}")
print(f"  F1-score : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

# Example: save the final model
# from joblib import dump
# dump(mlp_final, "mlp_anova500_tuned_final.joblib")

X shape: (473, 239)
Class distribution: [104 369]

Tuned MLP performance (5-fold OOF using fixed best params):
  Accuracy : 0.9387
  Precision: 0.9497
  Recall   : 0.9729
  F1-score : 0.9612

Confusion matrix:
[[ 85  19]
 [ 10 359]]

Classification report:
              precision    recall  f1-score   support

           0     0.8947    0.8173    0.8543       104
           1     0.9497    0.9729    0.9612       369

    accuracy                         0.9387       473
   macro avg     0.9222    0.8951    0.9077       473
weighted avg     0.9376    0.9387    0.9377       473


Fold-wise mean ± std with fixed tuned MLP:
  Accuracy : 0.9387 ± 0.0196
  Precision: 0.9524 ± 0.0177
  Recall   : 0.9703 ± 0.0179
  F1-score : 0.9611 ± 0.0122


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =====================================================
# 1. Load data
# =====================================================
data = pd.read_csv("Final Genes (XGB, LR, SVC).csv")

# Features: all except TumorType
X = data.drop(columns=["TumorType"], errors="ignore")
# Labels: binary 0/1 (Metastatic vs others)
y = (data["TumorType"])

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =====================================================
# 2. Fixed MLP with your best hyperparameters
# =====================================================
best_params = {
    'activation': 'relu',
    'alpha': 0.0001,
    'batch_size': 64,
    'hidden_layer_sizes': (256, 128),
    'learning_rate_init': 0.0005,
    'solver': 'adam'
}

mlp_best = MLPClassifier(
    **best_params,
    max_iter=400,
    random_state=42
)

# =====================================================
# 3. 5-fold CV evaluation with these parameters
# =====================================================
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions to compute global metrics
y_pred_oof = cross_val_predict(
    mlp_best,
    X,
    y,
    cv=cv_outer,
    n_jobs=-1
)

acc = accuracy_score(y, y_pred_oof)
prec = precision_score(y, y_pred_oof, average="binary", zero_division=0)
rec = recall_score(y, y_pred_oof, average="binary", zero_division=0)
f1 = f1_score(y, y_pred_oof, average="binary", zero_division=0)
cm = confusion_matrix(y, y_pred_oof)

print("\nTuned MLP performance (5-fold OOF using fixed best params):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")
print("\nConfusion matrix:")
print(cm)
print("\nClassification report:")
print(classification_report(y, y_pred_oof, digits=4))

# Also show fold-wise stats if you want mean ± std
fold_accs, fold_precs, fold_recs, fold_f1s = [], [], [], []

for fold_id, (train_idx, test_idx) in enumerate(cv_outer.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf = MLPClassifier(
        **best_params,
        max_iter=400,
        random_state=42 + fold_id  # slightly vary seed across folds
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    fold_accs.append(accuracy_score(y_test, y_pred))
    fold_precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
    fold_recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
    fold_f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

print("\nFold-wise mean ± std with fixed tuned MLP:")
print(f"  Accuracy : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
print(f"  Precision: {np.mean(fold_precs):.4f} ± {np.std(fold_precs):.4f}")
print(f"  Recall   : {np.mean(fold_recs):.4f} ± {np.std(fold_recs):.4f}")
print(f"  F1-score : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

# Example: save the final model
# from joblib import dump
# dump(mlp_final, "mlp_anova500_tuned_final.joblib")

X shape: (473, 27)
Class distribution: [104 369]

Tuned MLP performance (5-fold OOF using fixed best params):
  Accuracy : 0.9302
  Precision: 0.9516
  Recall   : 0.9593
  F1-score : 0.9555

Confusion matrix:
[[ 86  18]
 [ 15 354]]

Classification report:
              precision    recall  f1-score   support

           0     0.8515    0.8269    0.8390       104
           1     0.9516    0.9593    0.9555       369

    accuracy                         0.9302       473
   macro avg     0.9015    0.8931    0.8972       473
weighted avg     0.9296    0.9302    0.9299       473


Fold-wise mean ± std with fixed tuned MLP:
  Accuracy : 0.9450 ± 0.0124
  Precision: 0.9599 ± 0.0077
  Recall   : 0.9702 ± 0.0198
  F1-score : 0.9649 ± 0.0082


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =====================================================
# 1. Load data
# =====================================================
data = pd.read_csv("SKCM_transposed.csv")

# Features: all except TumorType
X = data.drop(columns=["index", "TumorType"], errors="ignore")
# Labels: binary 0/1 (Metastatic vs others)
y = (data["TumorType"] == "Metastatic").astype(int)

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =====================================================
# 2. Fixed MLP with your best hyperparameters
# =====================================================
best_params = {
    'activation': 'relu',
    'alpha': 0.0001,
    'batch_size': 64,
    'hidden_layer_sizes': (256, 128),
    'learning_rate_init': 0.0005,
    'solver': 'adam'
}

mlp_best = MLPClassifier(
    **best_params,
    max_iter=400,
    random_state=42
)

# =====================================================
# 3. 5-fold CV evaluation with these parameters
# =====================================================
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions to compute global metrics
y_pred_oof = cross_val_predict(
    mlp_best,
    X,
    y,
    cv=cv_outer,
    n_jobs=-1
)

acc = accuracy_score(y, y_pred_oof)
prec = precision_score(y, y_pred_oof, average="binary", zero_division=0)
rec = recall_score(y, y_pred_oof, average="binary", zero_division=0)
f1 = f1_score(y, y_pred_oof, average="binary", zero_division=0)
cm = confusion_matrix(y, y_pred_oof)

print("\nTuned MLP performance (5-fold OOF using fixed best params):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")
print("\nConfusion matrix:")
print(cm)
print("\nClassification report:")
print(classification_report(y, y_pred_oof, digits=4))

# Also show fold-wise stats if you want mean ± std
fold_accs, fold_precs, fold_recs, fold_f1s = [], [], [], []

for fold_id, (train_idx, test_idx) in enumerate(cv_outer.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf = MLPClassifier(
        **best_params,
        max_iter=400,
        random_state=42 + fold_id  # slightly vary seed across folds
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    fold_accs.append(accuracy_score(y_test, y_pred))
    fold_precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
    fold_recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
    fold_f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

print("\nFold-wise mean ± std with fixed tuned MLP:")
print(f"  Accuracy : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
print(f"  Precision: {np.mean(fold_precs):.4f} ± {np.std(fold_precs):.4f}")
print(f"  Recall   : {np.mean(fold_recs):.4f} ± {np.std(fold_recs):.4f}")
print(f"  F1-score : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

# Example: save the final model
# from joblib import dump
# dump(mlp_final, "mlp_anova500_tuned_final.joblib")

X shape: (473, 20530)
Class distribution: [104 369]

Tuned MLP performance (5-fold OOF using fixed best params):
  Accuracy : 0.8816
  Precision: 0.8864
  Recall   : 0.9729
  F1-score : 0.9276

Confusion matrix:
[[ 58  46]
 [ 10 359]]

Classification report:
              precision    recall  f1-score   support

           0     0.8529    0.5577    0.6744       104
           1     0.8864    0.9729    0.9276       369

    accuracy                         0.8816       473
   macro avg     0.8697    0.7653    0.8010       473
weighted avg     0.8791    0.8816    0.8720       473


Fold-wise mean ± std with fixed tuned MLP:
  Accuracy : 0.8857 ± 0.0275
  Precision: 0.8844 ± 0.0299
  Recall   : 0.9838 ± 0.0199
  F1-score : 0.9310 ± 0.0153


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =====================================================
# 1. Load data
# =====================================================
data = pd.read_csv("Final_RF.csv")

# Features: all except TumorType
X = data.drop(columns=["TumorType"], errors="ignore")
# Labels: binary 0/1 (Metastatic vs others)
y = (data["TumorType"])

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =====================================================
# 2. Define MLP and hyperparameter grid
# =====================================================
mlp = MLPClassifier(max_iter=400, random_state=42)

# Start with a *moderate* grid first. You can expand later if runtime is OK.
param_grid = {
    "hidden_layer_sizes": [
        (64,),         # 1 layer
        (128, 64),     # 2 layers
        (256, 128),    # deeper
    ],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [1e-4, 1e-3],          # L2 regularization
    "learning_rate_init": [1e-3, 5e-4],
    "batch_size": [32, 64]
}

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring="accuracy",
    n_jobs=-1,
    cv=cv_inner,
    verbose=1
)

print("\nRunning GridSearchCV for MLP hyperparameters...\n")
grid.fit(X, y)

print("\nBest parameters from GridSearchCV:")
print(grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.4f}")

best_params = grid.best_params_

# =====================================================
# 3. Evaluate MLP with best hyperparameters (outer CV)
# =====================================================
print("\nEvaluating tuned MLP with 5-fold cross-validation...\n")

mlp_best = MLPClassifier(
    **best_params,
    max_iter=400,
    random_state=42
)

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get out-of-fold predictions for metrics
y_pred_oof = cross_val_predict(
    mlp_best,
    X,
    y,
    cv=cv_outer,
    n_jobs=-1
)

acc = accuracy_score(y, y_pred_oof)
prec = precision_score(y, y_pred_oof, average="binary", zero_division=0)
rec = recall_score(y, y_pred_oof, average="binary", zero_division=0)
f1 = f1_score(y, y_pred_oof, average="binary", zero_division=0)
cm = confusion_matrix(y, y_pred_oof)

print("Overall performance with tuned MLP (5-fold OOF):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")
print("\nConfusion matrix:")
print(cm)
print("\nClassification report:")
print(classification_report(y, y_pred_oof, digits=4))

X shape: (473, 239)
Class distribution: [104 369]

Running GridSearchCV for MLP hyperparameters...

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters from GridSearchCV:
{'activation': 'relu', 'alpha': 0.001, 'batch_size': 64, 'hidden_layer_sizes': (128, 64), 'learning_rate_init': 0.001, 'solver': 'adam'}
Best CV accuracy: 0.9514

Evaluating tuned MLP with 5-fold cross-validation...

Overall performance with tuned MLP (5-fold OOF):
  Accuracy : 0.9514
  Precision: 0.9601
  Recall   : 0.9783
  F1-score : 0.9691

Confusion matrix:
[[ 89  15]
 [  8 361]]

Classification report:
              precision    recall  f1-score   support

           0     0.9175    0.8558    0.8856       104
           1     0.9601    0.9783    0.9691       369

    accuracy                         0.9514       473
   macro avg     0.9388    0.9170    0.9273       473
weighted avg     0.9507    0.9514    0.9508       473



In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =====================================================
# 1. Load data
# =====================================================
data = pd.read_csv("Final Genes (XGB, LR, SVC).csv")

# Features: all except TumorType
X = data.drop(columns=["TumorType"], errors="ignore")
# Labels: binary 0/1 (Metastatic vs others)
y = (data["TumorType"])

print("X shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =====================================================
# 2. Define MLP and hyperparameter grid
# =====================================================
mlp = MLPClassifier(max_iter=400, random_state=42)

# Start with a *moderate* grid first. You can expand later if runtime is OK.
param_grid = {
    "hidden_layer_sizes": [
        (64,),         # 1 layer
        (128, 64),     # 2 layers
        (256, 128),    # deeper
    ],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [1e-4, 1e-3],          # L2 regularization
    "learning_rate_init": [1e-3, 5e-4],
    "batch_size": [32, 64]
}

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring="accuracy",
    n_jobs=-1,
    cv=cv_inner,
    verbose=1
)

print("\nRunning GridSearchCV for MLP hyperparameters...\n")
grid.fit(X, y)

print("\nBest parameters from GridSearchCV:")
print(grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.4f}")

best_params = grid.best_params_

# =====================================================
# 3. Evaluate MLP with best hyperparameters (outer CV)
# =====================================================
print("\nEvaluating tuned MLP with 5-fold cross-validation...\n")

mlp_best = MLPClassifier(
    **best_params,
    max_iter=400,
    random_state=42
)

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get out-of-fold predictions for metrics
y_pred_oof = cross_val_predict(
    mlp_best,
    X,
    y,
    cv=cv_outer,
    n_jobs=-1
)

acc = accuracy_score(y, y_pred_oof)
prec = precision_score(y, y_pred_oof, average="binary", zero_division=0)
rec = recall_score(y, y_pred_oof, average="binary", zero_division=0)
f1 = f1_score(y, y_pred_oof, average="binary", zero_division=0)
cm = confusion_matrix(y, y_pred_oof)

print("Overall performance with tuned MLP (5-fold OOF):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")
print("\nConfusion matrix:")
print(cm)
print("\nClassification report:")
print(classification_report(y, y_pred_oof, digits=4))

X shape: (473, 239)
Class distribution: [473]

Running GridSearchCV for MLP hyperparameters...

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters from GridSearchCV:
{'activation': 'relu', 'alpha': 0.0001, 'batch_size': 32, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001, 'solver': 'adam'}
Best CV accuracy: 1.0000

Evaluating tuned MLP with 5-fold cross-validation...

Overall performance with tuned MLP (5-fold OOF):
  Accuracy : 1.0000
  Precision: 0.0000
  Recall   : 0.0000
  F1-score : 0.0000

Confusion matrix:
[[473]]

Classification report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       473

    accuracy                         1.0000       473
   macro avg     1.0000    1.0000    1.0000       473
weighted avg     1.0000    1.0000    1.0000       473



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('SKCM_transposed.csv')
genes = ['KRT6B', 'EBF2', 'C7', 'STMN2', 'KRT80', 'MMP3', 'CDK14', 'FAT2', 'TNFSF11', 'CDHR1','ALDH3B2', 'A2ML1', 'OVOL1', 'MASP1', 'DNAJC5B', 'S100A7', 'MMP1', 'LGALS7B',
'PTPRC', 'DMKN', 'DPYD', 'C10orf12', 'LOC643008', 'DIO2', 'DNAH17', 'KLK5', 'IVL']

X = df[genes]
y = (df['TumorType'] == 'Metastatic').astype(int)
print(X.head())

     KRT6B    EBF2      C7   STMN2   KRT80    MMP3    CDK14    FAT2  TNFSF11  \
0   0.0000  0.0000  5.6681  6.8125  0.5195  0.9006  10.9263  1.8483   2.0123   
1   3.7468  2.2997  9.4089  0.7259  2.7835  0.7259   4.4545  7.9361   4.1167   
2  10.1128  2.3509  1.0831  6.8606  4.2683  6.8652   5.4798  7.0430   0.4572   
3   2.2529  3.2441  6.3900  0.0000  4.5019  1.9354   8.7240  5.0033   3.8261   
4   2.1144  1.8734  1.9989  5.6034  1.5840  6.1210   8.7534  6.3852   1.7360   

    CDHR1  ...  LGALS7B    PTPRC    DMKN    DPYD  C10orf12  LOC643008    DIO2  \
0  0.5195  ...   0.0000   6.2803  0.5195  4.1968    5.8727     1.8483  2.4154   
1  4.0613  ...   1.2065   9.0084  8.8061  7.4125    5.2081     2.6395  2.0941   
2  7.6009  ...   9.2855   8.1172  9.3570  6.4451    4.4483     4.0583  8.5173   
3  2.9243  ...   0.0000  10.9012  9.3563  8.7336    6.0928     0.9572  3.9188   
4  0.7364  ...   0.0000   7.5143  8.2481  6.1279    5.4671     2.5838  8.8751   

   DNAH17    KLK5     IVL  
0  5

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------------------------------
# 1. Define MLP and Transformer classifiers
# -----------------------------------------------------

# MLP (scikit-learn)
classifiers = {
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10
    )
    # Transformer is handled separately below
}

# Transformer setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class SimpleTransformerClassifier(nn.Module):
    """
    Small transformer treating each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=1, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        self.input_proj = nn.Linear(1, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)           # (b, f, 1)
        x = self.input_proj(x)               # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :] # add positional embedding
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)       # (b, 1+f, d_model)
        x = self.encoder(x)                  # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                 # (b, d_model)
        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_once(
    X_train,
    y_train,
    n_epochs=40,
    batch_size=32,
    lr=1e-3,
    d_model=64,
    nhead=4,
    num_layers=1
):
    """
    Train transformer on a given train split with internal val split.
    """
    from sklearn.model_selection import train_test_split

    y_train = pd.Series(y_train).astype(int)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=0
    )

    n_features = X_tr.shape[1]
    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X_tr_t = torch.tensor(X_tr.values, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.values.astype(np.float32))
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values.astype(np.float32))

    tr_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds


# -----------------------------------------------------
# 2. Cross-validation (same as your code)
# -----------------------------------------------------

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

# ---------- MLP ----------
for clf_name, clf in classifiers.items():
    print(f"\n=== Running {clf_name} ===")
    fold_metrics = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\nProcessing Fold {fold+1}/{n_splits}")
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        # Handle binary/multiclass cases for metrics
        if len(np.unique(y)) == 2:
            avg_type = "binary"
        else:
            avg_type = "weighted"

        prec = precision_score(y_test, y_pred, average=avg_type, zero_division=0)
        rec = recall_score(y_test, y_pred, average=avg_type, zero_division=0)
        f1 = f1_score(y_test, y_pred, average=avg_type, zero_division=0)
        cm = confusion_matrix(y_test, y_pred)

        print(f"Fold {fold+1} Test: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")
        print(f"Confusion Matrix for Fold {fold+1}:\n{cm}\n")

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold+1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
            "Confusion Matrix": cm.tolist()
        })

    # Compute average and std for each metric
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]
    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
        "Confusion Matrix": ""
    }
    fold_metrics.append(avg_results)
    results.extend(fold_metrics)

# ---------- Transformer ----------
clf_name = "Transformer"
print(f"\n=== Running {clf_name} ===")
fold_metrics = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nProcessing Fold {fold+1}/{n_splits}")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # train transformer on training fold
    model_tr = train_transformer_once(
        X_train,
        y_train,
        n_epochs=40,
        batch_size=32,
        lr=1e-3,
        d_model=64,
        nhead=4,
        num_layers=1
    )
    y_pred = predict_transformer(model_tr, X_test)

    acc = accuracy_score(y_test, y_pred)
    if len(np.unique(y)) == 2:
        avg_type = "binary"
    else:
        avg_type = "weighted"

    prec = precision_score(y_test, y_pred, average=avg_type, zero_division=0)
    rec = recall_score(y_test, y_pred, average=avg_type, zero_division=0)
    f1 = f1_score(y_test, y_pred, average=avg_type, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Fold {fold+1} Test: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")
    print(f"Confusion Matrix for Fold {fold+1}:\n{cm}\n")

    fold_metrics.append({
        "Classifier": clf_name,
        "Fold": fold+1,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "Confusion Matrix": cm.tolist()
    })

# averages for Transformer
accs = [m["Accuracy"] for m in fold_metrics]
precs = [m["Precision"] for m in fold_metrics]
recs = [m["Recall"] for m in fold_metrics]
f1s = [m["F1-score"] for m in fold_metrics]
avg_results = {
    "Classifier": clf_name,
    "Fold": "Average",
    "Accuracy": np.mean(accs),
    "Accuracy_std": np.std(accs),
    "Precision": np.mean(precs),
    "Precision_std": np.std(precs),
    "Recall": np.mean(recs),
    "Recall_std": np.std(recs),
    "F1-score": np.mean(f1s),
    "F1-score_std": np.std(f1s),
    "Confusion Matrix": ""
}
fold_metrics.append(avg_results)
results.extend(fold_metrics)

# -----------------------------------------------------
# 3. Save and print summary (same pattern as original)
# -----------------------------------------------------

results_df = pd.DataFrame(results)
results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nAll results saved to mlp_transformer_cv_results.csv")

# --- Print mean ± std summary for each classifier ---
print("\nClassifier Performance Summary (mean ± std over 5 folds):\n")
for clf_name in ["MLP", "Transformer"]:
    avg_row = results_df[(results_df["Classifier"] == clf_name) & (results_df["Fold"] == "Average")]

    # Sum confusion matrices from each fold
    fold_rows = results_df[(results_df["Classifier"] == clf_name) & (results_df["Fold"] != "Average")]
    if not fold_rows.empty:
        # initialize cm_sum with zeros in appropriate shape
        first_cm = np.array(fold_rows.iloc[0]["Confusion Matrix"])
        cm_sum = np.zeros_like(first_cm, dtype=int)
        for _, row in fold_rows.iterrows():
            cm = np.array(row["Confusion Matrix"])
            cm_sum += cm
    else:
        cm_sum = np.array([])

    if not avg_row.empty:
        acc = avg_row["Accuracy"].values[0]
        acc_std = avg_row["Accuracy_std"].values[0]
        prec = avg_row["Precision"].values[0]
        prec_std = avg_row["Precision_std"].values[0]
        rec = avg_row["Recall"].values[0]
        rec_std = avg_row["Recall_std"].values[0]
        f1 = avg_row["F1-score"].values[0]
        f1_std = avg_row["F1-score_std"].values[0]
        print(f"Algorithm: {clf_name}")
        print(f"  Accuracy:  {acc:.4f} ± {acc_std:.4f}")
        print(f"  Precision: {prec:.4f} ± {prec_std:.4f}")
        print(f"  Recall:    {rec:.4f} ± {rec_std:.4f}")
        print(f"  F1 Score:  {f1:.4f} ± {f1_std:.4f}")
        print("  Confusion Matrix (summed over folds):")
        print(cm_sum)
        print("-" * 60)

Using device: cuda

=== Running MLP ===

Processing Fold 1/5
Fold 1 Test: Acc=0.9053, Prec=0.9114, Rec=0.9730, F1=0.9412
Confusion Matrix for Fold 1:
[[14  7]
 [ 2 72]]


Processing Fold 2/5
Fold 2 Test: Acc=0.8842, Prec=0.9091, Rec=0.9459, F1=0.9272
Confusion Matrix for Fold 2:
[[14  7]
 [ 4 70]]


Processing Fold 3/5
Fold 3 Test: Acc=0.8632, Prec=0.8675, Rec=0.9730, F1=0.9172
Confusion Matrix for Fold 3:
[[10 11]
 [ 2 72]]


Processing Fold 4/5
Fold 4 Test: Acc=0.8830, Prec=0.8889, Rec=0.9730, F1=0.9290
Confusion Matrix for Fold 4:
[[11  9]
 [ 2 72]]


Processing Fold 5/5
Fold 5 Test: Acc=0.9255, Prec=0.9125, Rec=1.0000, F1=0.9542
Confusion Matrix for Fold 5:
[[14  7]
 [ 0 73]]


=== Running Transformer ===

Processing Fold 1/5
Fold 1 Test: Acc=0.9158, Prec=0.9342, Rec=0.9595, F1=0.9467
Confusion Matrix for Fold 1:
[[16  5]
 [ 3 71]]


Processing Fold 2/5
Fold 2 Test: Acc=0.9053, Prec=0.9577, Rec=0.9189, F1=0.9379
Confusion Matrix for Fold 2:
[[18  3]
 [ 6 68]]


Processing Fold 3/5


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# =====================================================
# 1. Load NSGA-II results and data
# =====================================================
df = pd.read_csv("Final_fCV_ParetoFront_ANOVA.csv")
data = pd.read_csv("SKCM_ANOVA_500.csv")  # your real data matrix

# binary target
y = (data["TumorType"] == 'Metastatic').astype(int)

# =====================================================
# 2. Define classifiers (including MLP; Transformer handled separately)
# =====================================================
classifiers_models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=None, min_samples_split=2, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "Gradient Boost": GradientBoostingClassifier(learning_rate=0.1, max_depth=5, n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(C=1, penalty='l1', solver='liblinear', max_iter=500),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, min_samples_split=5, max_depth=None, random_state=42),
    "SVC": SVC(C=0.1, class_weight=None, gamma='scale', kernel='linear', probability=True),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=11, use_label_encoder=False, eval_metric='logloss'),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10
    ),
    # "Transformer" will be handled in a special branch below
}

folds = ["F1", "F2", "F3", "F4", "F5"]
classifier_keys = ["AB", "DT", "ET", "GB", "KNN", "LR", "NB", "RF", "SVC", "XGB", "MLP", "TF"]

# =====================================================
# 3. Small transformer model for tabular data
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class SimpleTransformerClassifier(nn.Module):
    """
    Small transformer treating each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=1, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        self.input_proj = nn.Linear(1, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)          # (b, f, 1)
        x = self.input_proj(x)              # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)      # (b, 1+f, d_model)
        x = self.encoder(x)                 # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                # (b, d_model)
        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_once(X_train, y_train,
                           n_epochs=50, batch_size=32, lr=1e-3,
                           d_model=64, nhead=4, num_layers=1):
    """
    Train a transformer once on a given train split.
    Uses internal 80/20 train/val split with early selection of best epoch.
    """
    from sklearn.model_selection import train_test_split

    y_train = pd.Series(y_train).astype(int)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=0
    )

    n_features = X_tr.shape[1]
    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X_tr_t = torch.tensor(X_tr.values, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.values.astype(np.float32))
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values.astype(np.float32))

    tr_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds


# =====================================================
# 4. Find common features per classifier (>=4 folds) as before
# =====================================================
common_features_dict = {}
for clf in classifier_keys:
    feature_counter = Counter()
    for fold in folds:
        rows = df[(df["Classifier"] == clf) & (df["Fold"] == fold)]
        fold_features = set()
        for feature_list in rows["Feature Names"]:
            features = [f.strip() for f in feature_list.split("|")]
            # Exclude '?' and '26823' and empty features
            features = [
                f for f in features
                if f and f != "?" and not all(c == "?" for c in f) and f != "26823"
            ]
            fold_features.update(features)
        feature_counter.update(fold_features)
    # Only keep features appearing in at least 4 folds
    common = [f for f, count in feature_counter.items() if count >= 4 and f != "26823"]
    common_features_dict[clf] = sorted(common)

# =====================================================
# 5. Evaluate all classifiers + Transformer on each subset
# =====================================================
for source_clf, feature_list in common_features_dict.items():
    # Remove any features not present in data
    valid_features = [f for f in feature_list if f in data.columns]
    num_features = len(valid_features)
    if not valid_features:
        print(f"Common features from {source_clf}(0): None found in >=4 folds or not in data.")
        continue

    X_sub = data[valid_features].copy()
    print(f"\nCommon features from {source_clf} ({num_features} features):")

    # ---- classical + MLP classifiers ----
    for clf_name, clf in classifiers_models.items():
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        accs, precs, recs, f1s = [], [], [], []
        for train_idx, test_idx in skf.split(X_sub, y):
            X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            try:
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
            except Exception as e:
                print(f"  {clf_name}: Error during training/evaluation: {e}")
                continue

            accs.append(accuracy_score(y_test, y_pred))
            precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
            recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
            f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

        print(
            f"  {clf_name}: "
            f"Accuracy = {np.mean(accs):.3f}±{np.std(accs):.3f}, "
            f"Precision = {np.mean(precs):.3f}±{np.std(precs):.3f}, "
            f"Recall = {np.mean(recs):.3f}±{np.std(recs):.3f}, "
            f"F1 Score = {np.mean(f1s):.3f}±{np.std(f1s):.3f}"
        )

    # ---- Transformer classifier ----
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, precs, recs, f1s = [], [], [], []
    for train_idx, test_idx in skf.split(X_sub, y):
        X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # train transformer on this fold
        model_tr = train_transformer_once(
            X_train,
            y_train,
            n_epochs=40,   # can adjust down if too slow
            batch_size=32,
            lr=1e-3,
            d_model=64,
            nhead=4,
            num_layers=1
        )
        y_pred = predict_transformer(model_tr, X_test)

        accs.append(accuracy_score(y_test, y_pred))
        precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
        recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
        f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

    print(
        f"  Transformer: "
        f"Accuracy = {np.mean(accs):.3f}±{np.std(accs):.3f}, "
        f"Precision = {np.mean(precs):.3f}±{np.std(precs):.3f}, "
        f"Recall = {np.mean(recs):.3f}±{np.std(recs):.3f}, "
        f"F1 Score = {np.mean(f1s):.3f}±{np.std(f1s):.3f}"
    )
    print()

Using device: cuda

Common features from AB (116 features):
  AdaBoost: Accuracy = 0.924±0.008, Precision = 0.926±0.015, Recall = 0.981±0.016, F1 Score = 0.953±0.005
  Decision Tree: Accuracy = 0.865±0.020, Precision = 0.896±0.017, Recall = 0.935±0.018, F1 Score = 0.915±0.013
  Extra Trees: Accuracy = 0.913±0.020, Precision = 0.919±0.016, Recall = 0.976±0.020, F1 Score = 0.946±0.013
  Gradient Boost: Accuracy = 0.918±0.010, Precision = 0.930±0.016, Recall = 0.968±0.018, F1 Score = 0.948±0.007
  KNN: Accuracy = 0.920±0.012, Precision = 0.932±0.014, Recall = 0.968±0.018, F1 Score = 0.949±0.008
  Logistic Regression: Accuracy = 0.898±0.009, Precision = 0.938±0.021, Recall = 0.932±0.024, F1 Score = 0.935±0.006
  Naive Bayes: Accuracy = 0.863±0.018, Precision = 0.886±0.008, Recall = 0.946±0.030, F1 Score = 0.915±0.013
  Random Forest: Accuracy = 0.907±0.018, Precision = 0.914±0.013, Recall = 0.973±0.017, F1 Score = 0.942±0.011
  SVC: Accuracy = 0.898±0.020, Precision = 0.935±0.015, Recall =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:28:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:28:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:28:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:28:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:28:09] WARNING: /w

  XGBoost: Accuracy = 0.924±0.010, Precision = 0.935±0.013, Recall = 0.970±0.016, F1 Score = 0.952±0.007
  MLP: Accuracy = 0.880±0.021, Precision = 0.898±0.012, Recall = 0.954±0.039, F1 Score = 0.925±0.015
  Transformer: Accuracy = 0.886±0.021, Precision = 0.900±0.029, Recall = 0.962±0.013, F1 Score = 0.930±0.012


Common features from DT (250 features):
  AdaBoost: Accuracy = 0.926±0.015, Precision = 0.929±0.018, Recall = 0.981±0.011, F1 Score = 0.954±0.009
  Decision Tree: Accuracy = 0.873±0.042, Precision = 0.915±0.031, Recall = 0.924±0.035, F1 Score = 0.919±0.027
  Extra Trees: Accuracy = 0.886±0.012, Precision = 0.887±0.010, Recall = 0.978±0.016, F1 Score = 0.930±0.007
  Gradient Boost: Accuracy = 0.909±0.020, Precision = 0.932±0.013, Recall = 0.954±0.035, F1 Score = 0.942±0.013
  KNN: Accuracy = 0.903±0.015, Precision = 0.918±0.012, Recall = 0.962±0.018, F1 Score = 0.939±0.010
  Logistic Regression: Accuracy = 0.901±0.029, Precision = 0.935±0.015, Recall = 0.938±0.028, F1 Score =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:29:37] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:29:38] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:29:39] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:29:39] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:29:40] WARNING: /w

  XGBoost: Accuracy = 0.928±0.008, Precision = 0.942±0.006, Recall = 0.967±0.007, F1 Score = 0.955±0.005
  MLP: Accuracy = 0.890±0.029, Precision = 0.912±0.026, Recall = 0.951±0.014, F1 Score = 0.931±0.018
  Transformer: Accuracy = 0.892±0.014, Precision = 0.894±0.017, Recall = 0.978±0.011, F1 Score = 0.934±0.008


Common features from ET (81 features):
  AdaBoost: Accuracy = 0.915±0.015, Precision = 0.923±0.017, Recall = 0.973±0.000, F1 Score = 0.947±0.009
  Decision Tree: Accuracy = 0.850±0.027, Precision = 0.907±0.013, Recall = 0.900±0.032, F1 Score = 0.903±0.018
  Extra Trees: Accuracy = 0.903±0.015, Precision = 0.903±0.013, Recall = 0.981±0.018, F1 Score = 0.940±0.010
  Gradient Boost: Accuracy = 0.909±0.010, Precision = 0.919±0.020, Recall = 0.970±0.016, F1 Score = 0.943±0.006
  KNN: Accuracy = 0.911±0.017, Precision = 0.918±0.014, Recall = 0.973±0.017, F1 Score = 0.945±0.011
  Logistic Regression: Accuracy = 0.894±0.018, Precision = 0.923±0.012, Recall = 0.943±0.022, F1 Score = 

...

  XGBoost: Accuracy = 0.928±0.014, Precision = 0.942±0.016, Recall = 0.968±0.022, F1 Score = 0.954±0.009
  MLP: Accuracy = 0.867±0.012, Precision = 0.883±0.016, Recall = 0.957±0.028, F1 Score = 0.918±0.009
  Transformer: Accuracy = 0.884±0.022, Precision = 0.903±0.029, Recall = 0.957±0.049, F1 Score = 0.927±0.015


Common features from MLP (137 features):
  AdaBoost: Accuracy = 0.911±0.010, Precision = 0.919±0.022, Recall = 0.973±0.021, F1 Score = 0.945±0.006
  Decision Tree: Accuracy = 0.888±0.024, Precision = 0.927±0.016, Recall = 0.930±0.020, F1 Score = 0.928±0.016
  Extra Trees: Accuracy = 0.896±0.008, Precision = 0.907±0.015, Recall = 0.968±0.016, F1 Score = 0.936±0.005
  Gradient Boost: Accuracy = 0.916±0.022, Precision = 0.930±0.013, Recall = 0.965±0.018, F1 Score = 0.947±0.014
  KNN: Accuracy = 0.903±0.024, Precision = 0.920±0.021, Recall = 0.959±0.024, F1 Score = 0.939±0.015
  Logistic Regression: Accuracy = 0.915±0.028, Precision = 0.951±0.021, Recall = 0.940±0.027, F1 Score 

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:05] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:05] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:06] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:06] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:06] WARNING: /w

  XGBoost: Accuracy = 0.920±0.017, Precision = 0.932±0.016, Recall = 0.967±0.022, F1 Score = 0.949±0.011
  MLP: Accuracy = 0.886±0.018, Precision = 0.910±0.019, Recall = 0.949±0.031, F1 Score = 0.928±0.012
  Transformer: Accuracy = 0.879±0.005, Precision = 0.893±0.017, Recall = 0.962±0.020, F1 Score = 0.926±0.003


Common features from TF (124 features):
  AdaBoost: Accuracy = 0.905±0.015, Precision = 0.903±0.011, Recall = 0.984±0.010, F1 Score = 0.942±0.009
  Decision Tree: Accuracy = 0.869±0.025, Precision = 0.925±0.016, Recall = 0.905±0.022, F1 Score = 0.915±0.016
  Extra Trees: Accuracy = 0.894±0.015, Precision = 0.894±0.014, Recall = 0.981±0.018, F1 Score = 0.935±0.009
  Gradient Boost: Accuracy = 0.922±0.011, Precision = 0.924±0.015, Recall = 0.981±0.007, F1 Score = 0.951±0.006
  KNN: Accuracy = 0.903±0.023, Precision = 0.922±0.019, Recall = 0.957±0.020, F1 Score = 0.939±0.014
  Logistic Regression: Accuracy = 0.909±0.020, Precision = 0.934±0.008, Recall = 0.951±0.022, F1 Score =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:53] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:53] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:37:54] WARNING: /w

  XGBoost: Accuracy = 0.920±0.009, Precision = 0.921±0.014, Recall = 0.981±0.007, F1 Score = 0.950±0.005
  MLP: Accuracy = 0.903±0.021, Precision = 0.910±0.021, Recall = 0.973±0.028, F1 Score = 0.940±0.014
  Transformer: Accuracy = 0.890±0.016, Precision = 0.900±0.022, Recall = 0.968±0.016, F1 Score = 0.932±0.009



In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# =====================================================
# 1. Load NSGA-II results and data
# =====================================================
df = pd.read_csv("Final_fCV_ParetoFront_ANOVA.csv")
data = pd.read_csv("SKCM_ANOVA_500.csv")  # your real data matrix

# binary target
y = (data["TumorType"] == 'Metastatic').astype(int)

# =====================================================
# 2. Define classifiers (including MLP; Transformer handled separately)
# =====================================================
classifiers_models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=None, min_samples_split=2, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "Gradient Boost": GradientBoostingClassifier(learning_rate=0.1, max_depth=5, n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(C=1, penalty='l1', solver='liblinear', max_iter=500),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, min_samples_split=5, max_depth=None, random_state=42),
    "SVC": SVC(C=0.1, class_weight=None, gamma='scale', kernel='linear', probability=True),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=11, use_label_encoder=False, eval_metric='logloss'),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10
    ),
    # "Transformer" will be handled in a special branch below
}

folds = ["F1", "F2", "F3", "F4", "F5"]
classifier_keys = ["AB", "DT", "ET", "GB", "KNN", "LR", "NB", "RF", "SVC", "XGB", "MLP", "TF"]

# =====================================================
# 3. Small transformer model for tabular data
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class SimpleTransformerClassifier(nn.Module):
    """
    Small transformer treating each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=1, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        self.input_proj = nn.Linear(1, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)          # (b, f, 1)
        x = self.input_proj(x)              # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)      # (b, 1+f, d_model)
        x = self.encoder(x)                 # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                # (b, d_model)
        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_once(X_train, y_train,
                           n_epochs=50, batch_size=32, lr=1e-3,
                           d_model=64, nhead=4, num_layers=1):
    """
    Train a transformer once on a given train split.
    Uses internal 80/20 train/val split with early selection of best epoch.
    """
    from sklearn.model_selection import train_test_split

    y_train = pd.Series(y_train).astype(int)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=0
    )

    n_features = X_tr.shape[1]
    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X_tr_t = torch.tensor(X_tr.values, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.values.astype(np.float32))
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values.astype(np.float32))

    tr_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds


# =====================================================
# 4. Find common features per classifier (>=4 folds) as before
# =====================================================
common_features_dict = {}
for clf in classifier_keys:
    feature_counter = Counter()
    for fold in folds:
        rows = df[(df["Classifier"] == clf) & (df["Fold"] == fold)]
        fold_features = set()
        for feature_list in rows["Feature Names"]:
            features = [f.strip() for f in feature_list.split("|")]
            # Exclude '?' and '26823' and empty features
            features = [
                f for f in features
                if f and f != "?" and not all(c == "?" for c in f) and f != "26823"
            ]
            fold_features.update(features)
        feature_counter.update(fold_features)
    # Only keep features appearing in at least 4 folds
    common = [f for f, count in feature_counter.items() if count >= 2 and f != "26823"]
    common_features_dict[clf] = sorted(common)

# =====================================================
# 5. Evaluate all classifiers + Transformer on each subset
# =====================================================
for source_clf, feature_list in common_features_dict.items():
    # Remove any features not present in data
    valid_features = [f for f in feature_list if f in data.columns]
    num_features = len(valid_features)
    if not valid_features:
        print(f"Common features from {source_clf}(0): None found in >=4 folds or not in data.")
        continue

    X_sub = data[valid_features].copy()
    print(f"\nCommon features from {source_clf} ({num_features} features):")

    # ---- classical + MLP classifiers ----
    for clf_name, clf in classifiers_models.items():
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        accs, precs, recs, f1s = [], [], [], []
        for train_idx, test_idx in skf.split(X_sub, y):
            X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            try:
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
            except Exception as e:
                print(f"  {clf_name}: Error during training/evaluation: {e}")
                continue

            accs.append(accuracy_score(y_test, y_pred))
            precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
            recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
            f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

        print(
            f"  {clf_name}: "
            f"Accuracy = {np.mean(accs):.3f}±{np.std(accs):.3f}, "
            f"Precision = {np.mean(precs):.3f}±{np.std(precs):.3f}, "
            f"Recall = {np.mean(recs):.3f}±{np.std(recs):.3f}, "
            f"F1 Score = {np.mean(f1s):.3f}±{np.std(f1s):.3f}"
        )

    # ---- Transformer classifier ----
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, precs, recs, f1s = [], [], [], []
    for train_idx, test_idx in skf.split(X_sub, y):
        X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # train transformer on this fold
        model_tr = train_transformer_once(
            X_train,
            y_train,
            n_epochs=40,   # can adjust down if too slow
            batch_size=32,
            lr=1e-3,
            d_model=64,
            nhead=4,
            num_layers=1
        )
        y_pred = predict_transformer(model_tr, X_test)

        accs.append(accuracy_score(y_test, y_pred))
        precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
        recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
        f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

    print(
        f"  Transformer: "
        f"Accuracy = {np.mean(accs):.3f}±{np.std(accs):.3f}, "
        f"Precision = {np.mean(precs):.3f}±{np.std(precs):.3f}, "
        f"Recall = {np.mean(recs):.3f}±{np.std(recs):.3f}, "
        f"F1 Score = {np.mean(f1s):.3f}±{np.std(f1s):.3f}"
    )
    print()

Using device: cuda

Common features from AB (431 features):
  AdaBoost: Accuracy = 0.922±0.011, Precision = 0.921±0.011, Recall = 0.984±0.010, F1 Score = 0.952±0.007
  Decision Tree: Accuracy = 0.880±0.023, Precision = 0.922±0.022, Recall = 0.924±0.025, F1 Score = 0.923±0.015
  Extra Trees: Accuracy = 0.894±0.015, Precision = 0.898±0.014, Recall = 0.976±0.016, F1 Score = 0.935±0.009
  Gradient Boost: Accuracy = 0.911±0.024, Precision = 0.927±0.017, Recall = 0.962±0.022, F1 Score = 0.944±0.016
  KNN: Accuracy = 0.911±0.018, Precision = 0.925±0.013, Recall = 0.965±0.024, F1 Score = 0.944±0.012
  Logistic Regression: Accuracy = 0.917±0.025, Precision = 0.948±0.015, Recall = 0.946±0.023, F1 Score = 0.947±0.016
  Naive Bayes: Accuracy = 0.856±0.022, Precision = 0.883±0.007, Recall = 0.940±0.033, F1 Score = 0.911±0.015
  Random Forest: Accuracy = 0.896±0.014, Precision = 0.907±0.021, Recall = 0.968±0.016, F1 Score = 0.936±0.008
  SVC: Accuracy = 0.909±0.025, Precision = 0.946±0.026, Recall =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:49:34] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:49:35] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:49:36] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:49:36] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:49:37] WARNING: /w

  XGBoost: Accuracy = 0.934±0.026, Precision = 0.943±0.023, Recall = 0.976±0.020, F1 Score = 0.959±0.016
  MLP: Accuracy = 0.860±0.036, Precision = 0.875±0.041, Recall = 0.962±0.022, F1 Score = 0.915±0.020
  Transformer: Accuracy = 0.873±0.018, Precision = 0.877±0.020, Recall = 0.976±0.018, F1 Score = 0.923±0.010


Common features from DT (488 features):
  AdaBoost: Accuracy = 0.918±0.015, Precision = 0.919±0.014, Recall = 0.981±0.011, F1 Score = 0.949±0.009
  Decision Tree: Accuracy = 0.854±0.025, Precision = 0.908±0.013, Recall = 0.905±0.036, F1 Score = 0.906±0.017
  Extra Trees: Accuracy = 0.894±0.019, Precision = 0.898±0.019, Recall = 0.976±0.016, F1 Score = 0.935±0.011
  Gradient Boost: Accuracy = 0.890±0.027, Precision = 0.912±0.015, Recall = 0.951±0.023, F1 Score = 0.931±0.017
  KNN: Accuracy = 0.905±0.022, Precision = 0.926±0.010, Recall = 0.954±0.025, F1 Score = 0.940±0.014
  Logistic Regression: Accuracy = 0.922±0.021, Precision = 0.959±0.009, Recall = 0.940±0.025, F1 Score =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:52:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:52:09] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:52:10] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:52:11] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:52:12] WARNING: /w

  XGBoost: Accuracy = 0.932±0.013, Precision = 0.934±0.020, Recall = 0.984±0.010, F1 Score = 0.958±0.008
  MLP: Accuracy = 0.871±0.022, Precision = 0.892±0.024, Recall = 0.951±0.043, F1 Score = 0.920±0.016
  Transformer: Accuracy = 0.882±0.020, Precision = 0.895±0.028, Recall = 0.962±0.022, F1 Score = 0.927±0.011


Common features from ET (390 features):
  AdaBoost: Accuracy = 0.918±0.018, Precision = 0.919±0.014, Recall = 0.981±0.014, F1 Score = 0.949±0.011
  Decision Tree: Accuracy = 0.884±0.043, Precision = 0.925±0.029, Recall = 0.927±0.031, F1 Score = 0.926±0.028
  Extra Trees: Accuracy = 0.896±0.012, Precision = 0.900±0.013, Recall = 0.976±0.020, F1 Score = 0.936±0.008
  Gradient Boost: Accuracy = 0.903±0.020, Precision = 0.913±0.014, Recall = 0.968±0.016, F1 Score = 0.940±0.013
  KNN: Accuracy = 0.909±0.019, Precision = 0.925±0.013, Recall = 0.962±0.022, F1 Score = 0.943±0.012
  Logistic Regression: Accuracy = 0.915±0.013, Precision = 0.946±0.017, Recall = 0.946±0.027, F1 Score =

...

  XGBoost: Accuracy = 0.941±0.011, Precision = 0.945±0.014, Recall = 0.981±0.014, F1 Score = 0.963±0.007
  MLP: Accuracy = 0.886±0.018, Precision = 0.928±0.018, Recall = 0.927±0.033, F1 Score = 0.927±0.013
  Transformer: Accuracy = 0.880±0.025, Precision = 0.887±0.023, Recall = 0.970±0.016, F1 Score = 0.926±0.014


Common features from MLP (457 features):
  AdaBoost: Accuracy = 0.920±0.012, Precision = 0.922±0.017, Recall = 0.981±0.011, F1 Score = 0.950±0.007
  Decision Tree: Accuracy = 0.871±0.031, Precision = 0.914±0.015, Recall = 0.921±0.030, F1 Score = 0.917±0.021
  Extra Trees: Accuracy = 0.896±0.018, Precision = 0.901±0.018, Recall = 0.976±0.020, F1 Score = 0.936±0.011
  Gradient Boost: Accuracy = 0.901±0.024, Precision = 0.924±0.015, Recall = 0.951±0.025, F1 Score = 0.937±0.016
  KNN: Accuracy = 0.903±0.022, Precision = 0.922±0.013, Recall = 0.957±0.026, F1 Score = 0.939±0.014
  Logistic Regression: Accuracy = 0.917±0.035, Precision = 0.956±0.011, Recall = 0.938±0.040, F1 Score 

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:13:31] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:13:32] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:13:33] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:13:35] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:13:36] WARNING: /w

  XGBoost: Accuracy = 0.930±0.013, Precision = 0.934±0.020, Recall = 0.981±0.014, F1 Score = 0.956±0.008
  MLP: Accuracy = 0.903±0.022, Precision = 0.919±0.030, Recall = 0.962±0.013, F1 Score = 0.939±0.013
  Transformer: Accuracy = 0.880±0.008, Precision = 0.879±0.010, Recall = 0.981±0.020, F1 Score = 0.927±0.006


Common features from TF (433 features):
  AdaBoost: Accuracy = 0.924±0.015, Precision = 0.924±0.013, Recall = 0.984±0.010, F1 Score = 0.953±0.009
  Decision Tree: Accuracy = 0.856±0.037, Precision = 0.902±0.036, Recall = 0.916±0.016, F1 Score = 0.909±0.023
  Extra Trees: Accuracy = 0.892±0.015, Precision = 0.896±0.015, Recall = 0.976±0.016, F1 Score = 0.934±0.009
  Gradient Boost: Accuracy = 0.890±0.038, Precision = 0.918±0.021, Recall = 0.943±0.031, F1 Score = 0.930±0.024
  KNN: Accuracy = 0.907±0.022, Precision = 0.927±0.010, Recall = 0.957±0.026, F1 Score = 0.941±0.014
  Logistic Regression: Accuracy = 0.928±0.026, Precision = 0.957±0.018, Recall = 0.951±0.023, F1 Score =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:15:56] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:15:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:15:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:15:59] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:16:01] WARNING: /w

  XGBoost: Accuracy = 0.939±0.014, Precision = 0.943±0.016, Recall = 0.981±0.014, F1 Score = 0.962±0.009
  MLP: Accuracy = 0.879±0.016, Precision = 0.897±0.021, Recall = 0.957±0.022, F1 Score = 0.925±0.010
  Transformer: Accuracy = 0.871±0.012, Precision = 0.878±0.014, Recall = 0.970±0.016, F1 Score = 0.922±0.007



In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# =====================================================
# 1. Load NSGA-II results and data
# =====================================================
df = pd.read_csv("Final_fCV_ParetoFront_ANOVA.csv")
data = pd.read_csv("SKCM_ANOVA_500.csv")  # your real data matrix

# binary target
y = (data["TumorType"] == 'Metastatic').astype(int)

# =====================================================
# 2. Define classifiers (including MLP; Transformer handled separately)
# =====================================================
classifiers_models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=None, min_samples_split=2, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "Gradient Boost": GradientBoostingClassifier(learning_rate=0.1, max_depth=5, n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(C=1, penalty='l1', solver='liblinear', max_iter=500),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, min_samples_split=5, max_depth=None, random_state=42),
    "SVC": SVC(C=0.1, class_weight=None, gamma='scale', kernel='linear', probability=True),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=11, use_label_encoder=False, eval_metric='logloss'),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10
    ),
    # "Transformer" will be handled in a special branch below
}

folds = ["F1", "F2", "F3", "F4", "F5"]
classifier_keys = ["AB", "DT", "ET", "GB", "KNN", "LR", "NB", "RF", "SVC", "XGB", "MLP", "TF"]

# =====================================================
# 3. Small transformer model for tabular data
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class SimpleTransformerClassifier(nn.Module):
    """
    Small transformer treating each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=1, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        self.input_proj = nn.Linear(1, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)          # (b, f, 1)
        x = self.input_proj(x)              # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)      # (b, 1+f, d_model)
        x = self.encoder(x)                 # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                # (b, d_model)
        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_once(X_train, y_train,
                           n_epochs=50, batch_size=32, lr=1e-3,
                           d_model=64, nhead=4, num_layers=1):
    """
    Train a transformer once on a given train split.
    Uses internal 80/20 train/val split with early selection of best epoch.
    """
    from sklearn.model_selection import train_test_split

    y_train = pd.Series(y_train).astype(int)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=0
    )

    n_features = X_tr.shape[1]
    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X_tr_t = torch.tensor(X_tr.values, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.values.astype(np.float32))
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values.astype(np.float32))

    tr_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds


# =====================================================
# 4. Find common features per classifier (>=4 folds) as before
# =====================================================
common_features_dict = {}
for clf in classifier_keys:
    feature_counter = Counter()
    for fold in folds:
        rows = df[(df["Classifier"] == clf) & (df["Fold"] == fold)]
        fold_features = set()
        for feature_list in rows["Feature Names"]:
            features = [f.strip() for f in feature_list.split("|")]
            # Exclude '?' and '26823' and empty features
            features = [
                f for f in features
                if f and f != "?" and not all(c == "?" for c in f) and f != "26823"
            ]
            fold_features.update(features)
        feature_counter.update(fold_features)
    # Only keep features appearing in at least 4 folds
    common = [f for f, count in feature_counter.items() if count >= 3 and f != "26823"]
    common_features_dict[clf] = sorted(common)

# =====================================================
# 5. Evaluate all classifiers + Transformer on each subset
# =====================================================
for source_clf, feature_list in common_features_dict.items():
    # Remove any features not present in data
    valid_features = [f for f in feature_list if f in data.columns]
    num_features = len(valid_features)
    if not valid_features:
        print(f"Common features from {source_clf}(0): None found in >=4 folds or not in data.")
        continue

    X_sub = data[valid_features].copy()
    print(f"\nCommon features from {source_clf} ({num_features} features):")

    # ---- classical + MLP classifiers ----
    for clf_name, clf in classifiers_models.items():
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        accs, precs, recs, f1s = [], [], [], []
        for train_idx, test_idx in skf.split(X_sub, y):
            X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            try:
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
            except Exception as e:
                print(f"  {clf_name}: Error during training/evaluation: {e}")
                continue

            accs.append(accuracy_score(y_test, y_pred))
            precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
            recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
            f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

        print(
            f"  {clf_name}: "
            f"Accuracy = {np.mean(accs):.3f}±{np.std(accs):.3f}, "
            f"Precision = {np.mean(precs):.3f}±{np.std(precs):.3f}, "
            f"Recall = {np.mean(recs):.3f}±{np.std(recs):.3f}, "
            f"F1 Score = {np.mean(f1s):.3f}±{np.std(f1s):.3f}"
        )

    # ---- Transformer classifier ----
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, precs, recs, f1s = [], [], [], []
    for train_idx, test_idx in skf.split(X_sub, y):
        X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # train transformer on this fold
        model_tr = train_transformer_once(
            X_train,
            y_train,
            n_epochs=40,   # can adjust down if too slow
            batch_size=32,
            lr=1e-3,
            d_model=64,
            nhead=4,
            num_layers=1
        )
        y_pred = predict_transformer(model_tr, X_test)

        accs.append(accuracy_score(y_test, y_pred))
        precs.append(precision_score(y_test, y_pred, average="binary", zero_division=0))
        recs.append(recall_score(y_test, y_pred, average="binary", zero_division=0))
        f1s.append(f1_score(y_test, y_pred, average="binary", zero_division=0))

    print(
        f"  Transformer: "
        f"Accuracy = {np.mean(accs):.3f}±{np.std(accs):.3f}, "
        f"Precision = {np.mean(precs):.3f}±{np.std(precs):.3f}, "
        f"Recall = {np.mean(recs):.3f}±{np.std(recs):.3f}, "
        f"F1 Score = {np.mean(f1s):.3f}±{np.std(f1s):.3f}"
    )
    print()

Using device: cuda

Common features from AB (279 features):
  AdaBoost: Accuracy = 0.920±0.012, Precision = 0.924±0.019, Recall = 0.978±0.014, F1 Score = 0.950±0.007
  Decision Tree: Accuracy = 0.875±0.031, Precision = 0.921±0.021, Recall = 0.919±0.024, F1 Score = 0.920±0.020
  Extra Trees: Accuracy = 0.894±0.017, Precision = 0.898±0.015, Recall = 0.976±0.020, F1 Score = 0.935±0.011
  Gradient Boost: Accuracy = 0.909±0.023, Precision = 0.923±0.015, Recall = 0.965±0.028, F1 Score = 0.943±0.015
  KNN: Accuracy = 0.909±0.016, Precision = 0.927±0.017, Recall = 0.959±0.024, F1 Score = 0.943±0.010
  Logistic Regression: Accuracy = 0.907±0.024, Precision = 0.951±0.023, Recall = 0.930±0.020, F1 Score = 0.940±0.015
  Naive Bayes: Accuracy = 0.858±0.020, Precision = 0.883±0.007, Recall = 0.943±0.031, F1 Score = 0.912±0.014
  Random Forest: Accuracy = 0.901±0.011, Precision = 0.907±0.014, Recall = 0.973±0.012, F1 Score = 0.939±0.006
  SVC: Accuracy = 0.888±0.025, Precision = 0.930±0.022, Recall =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:55:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:55:55] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:55:55] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:55:56] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:55:58] WARNING: /w

  XGBoost: Accuracy = 0.924±0.017, Precision = 0.930±0.015, Recall = 0.976±0.018, F1 Score = 0.952±0.011
  MLP: Accuracy = 0.873±0.027, Precision = 0.900±0.017, Recall = 0.943±0.039, F1 Score = 0.920±0.019
  Transformer: Accuracy = 0.886±0.012, Precision = 0.898±0.025, Recall = 0.965±0.022, F1 Score = 0.930±0.007


Common features from DT (397 features):
  AdaBoost: Accuracy = 0.920±0.011, Precision = 0.921±0.013, Recall = 0.981±0.011, F1 Score = 0.950±0.007
  Decision Tree: Accuracy = 0.861±0.035, Precision = 0.925±0.020, Recall = 0.894±0.033, F1 Score = 0.909±0.023
  Extra Trees: Accuracy = 0.888±0.014, Precision = 0.895±0.010, Recall = 0.970±0.020, F1 Score = 0.931±0.009
  Gradient Boost: Accuracy = 0.901±0.028, Precision = 0.924±0.020, Recall = 0.951±0.020, F1 Score = 0.937±0.018
  KNN: Accuracy = 0.886±0.015, Precision = 0.909±0.007, Recall = 0.949±0.020, F1 Score = 0.928±0.010
  Logistic Regression: Accuracy = 0.913±0.018, Precision = 0.946±0.015, Recall = 0.943±0.026, F1 Score =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:58:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:58:02] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:58:03] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:58:04] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:58:05] WARNING: /w

  XGBoost: Accuracy = 0.932±0.020, Precision = 0.938±0.023, Recall = 0.978±0.011, F1 Score = 0.958±0.012
  MLP: Accuracy = 0.882±0.031, Precision = 0.911±0.019, Recall = 0.940±0.031, F1 Score = 0.925±0.021
  Transformer: Accuracy = 0.869±0.011, Precision = 0.874±0.010, Recall = 0.973±0.015, F1 Score = 0.921±0.007


Common features from ET (240 features):
  AdaBoost: Accuracy = 0.922±0.012, Precision = 0.928±0.016, Recall = 0.976±0.018, F1 Score = 0.951±0.008
  Decision Tree: Accuracy = 0.871±0.033, Precision = 0.907±0.019, Recall = 0.930±0.031, F1 Score = 0.918±0.021
  Extra Trees: Accuracy = 0.899±0.014, Precision = 0.901±0.017, Recall = 0.978±0.016, F1 Score = 0.938±0.009
  Gradient Boost: Accuracy = 0.894±0.043, Precision = 0.910±0.022, Recall = 0.959±0.038, F1 Score = 0.934±0.028
  KNN: Accuracy = 0.903±0.015, Precision = 0.915±0.011, Recall = 0.965±0.025, F1 Score = 0.939±0.010
  Logistic Regression: Accuracy = 0.894±0.019, Precision = 0.935±0.018, Recall = 0.930±0.026, F1 Score =

...

  XGBoost: Accuracy = 0.941±0.011, Precision = 0.948±0.013, Recall = 0.978±0.011, F1 Score = 0.963±0.007
  MLP: Accuracy = 0.897±0.023, Precision = 0.908±0.028, Recall = 0.968±0.034, F1 Score = 0.936±0.014
  Transformer: Accuracy = 0.890±0.017, Precision = 0.892±0.018, Recall = 0.978±0.011, F1 Score = 0.933±0.010


Common features from MLP (313 features):
  AdaBoost: Accuracy = 0.926±0.015, Precision = 0.931±0.012, Recall = 0.978±0.016, F1 Score = 0.954±0.009
  Decision Tree: Accuracy = 0.861±0.018, Precision = 0.915±0.013, Recall = 0.905±0.031, F1 Score = 0.910±0.013
  Extra Trees: Accuracy = 0.899±0.016, Precision = 0.902±0.013, Recall = 0.976±0.016, F1 Score = 0.938±0.010
  Gradient Boost: Accuracy = 0.907±0.015, Precision = 0.924±0.014, Recall = 0.959±0.012, F1 Score = 0.942±0.010
  KNN: Accuracy = 0.907±0.017, Precision = 0.924±0.004, Recall = 0.959±0.024, F1 Score = 0.941±0.011
  Logistic Regression: Accuracy = 0.913±0.027, Precision = 0.953±0.008, Recall = 0.935±0.030, F1 Score 

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:12:34] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:12:36] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:12:37] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:12:37] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:12:38] WARNING: /w

  XGBoost: Accuracy = 0.932±0.009, Precision = 0.940±0.012, Recall = 0.976±0.016, F1 Score = 0.957±0.006
  MLP: Accuracy = 0.892±0.045, Precision = 0.931±0.029, Recall = 0.932±0.056, F1 Score = 0.930±0.030
  Transformer: Accuracy = 0.873±0.018, Precision = 0.879±0.025, Recall = 0.973±0.023, F1 Score = 0.923±0.010


Common features from TF (283 features):
  AdaBoost: Accuracy = 0.918±0.012, Precision = 0.922±0.020, Recall = 0.978±0.011, F1 Score = 0.949±0.007
  Decision Tree: Accuracy = 0.823±0.031, Precision = 0.903±0.024, Recall = 0.867±0.044, F1 Score = 0.884±0.023
  Extra Trees: Accuracy = 0.886±0.012, Precision = 0.891±0.017, Recall = 0.973±0.019, F1 Score = 0.930±0.007
  Gradient Boost: Accuracy = 0.913±0.015, Precision = 0.923±0.016, Recall = 0.970±0.020, F1 Score = 0.946±0.010
  KNN: Accuracy = 0.896±0.021, Precision = 0.922±0.020, Recall = 0.949±0.023, F1 Score = 0.935±0.013
  Logistic Regression: Accuracy = 0.918±0.016, Precision = 0.944±0.018, Recall = 0.951±0.018, F1 Score =

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:14:11] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:14:12] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:14:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:14:14] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:14:14] WARNING: /w

  XGBoost: Accuracy = 0.920±0.012, Precision = 0.924±0.016, Recall = 0.978±0.011, F1 Score = 0.950±0.007
  MLP: Accuracy = 0.920±0.023, Precision = 0.930±0.022, Recall = 0.970±0.022, F1 Score = 0.950±0.014
  Transformer: Accuracy = 0.871±0.012, Precision = 0.882±0.013, Recall = 0.965±0.016, F1 Score = 0.921±0.007



In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# --- Load CSVs ---
df = pd.read_csv("Final_fCV_ParetoFront_ANOVA.csv")
data = pd.read_csv("SKCM_ANOVA_500.csv")

# --- Find features common in at least 3 folds for Random Forest (RF) ---
folds = ["F1", "F2", "F3", "F4", "F5"]
rf_feature_counter = Counter()
for fold in folds:
    rows = df[(df["Classifier"] == "RF") & (df["Fold"] == fold)]
    fold_features = set()
    for feature_list in rows["Feature Names"]:
        features = [f.strip() for f in feature_list.split("|")]
        features = [f for f in features if f and f != '?' and not all(c == '?' for c in f) and f != '26823']
        fold_features.update(features)
    rf_feature_counter.update(fold_features)

# Only keep features appearing in at least 3 folds
rf_common_features = sorted([f for f, count in rf_feature_counter.items() if count >= 3 and f != '26823'])

# --- Filter Data ---
valid_features = [f for f in rf_common_features if f in data.columns]
num_features = len(valid_features)
if not valid_features:
    raise ValueError("No features from RF found in >=3 folds and present in the data.")

X = data[valid_features].copy()
y = (data["TumorType"] == 'Metastatic').astype(int)

print(f"Using {num_features} features common in at least 3 folds for Random Forest.")

# =====================================================
# 1. Define MLP and Transformer
# =====================================================

# MLP (scikit-learn)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=300,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=10
)

# Small transformer for tabular data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class SimpleTransformerClassifier(nn.Module):
    """
    Small transformer treating each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=1, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        self.input_proj = nn.Linear(1, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)          # (b, f, 1)
        x = self.input_proj(x)              # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)      # (b, 1+f, d_model)
        x = self.encoder(x)                 # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                # (b, d_model)
        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_once(
    X_train,
    y_train,
    n_epochs=40,
    batch_size=32,
    lr=1e-3,
    d_model=64,
    nhead=4,
    num_layers=1
):
    """
    Train transformer on a given train split with internal val split.
    """
    from sklearn.model_selection import train_test_split

    y_train = pd.Series(y_train).astype(int)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=0
    )

    n_features = X_tr.shape[1]
    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X_tr_t = torch.tensor(X_tr.values, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.values.astype(np.float32))
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values.astype(np.float32))

    tr_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds


# =====================================================
# 2. Cross-Validation: MLP and Transformer only
# =====================================================

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

# ---------- MLP ----------
clf_name = "MLP"
clf = mlp_model
print(f"\n=== Running {clf_name} ===")
fold_metrics = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nProcessing Fold {fold+1}/{n_splits}")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    clf.fit(X_train_fold, y_train_fold)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
    rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Fold {fold+1} Test: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")
    print(f"Confusion Matrix for Fold {fold+1}:\n{cm}\n")

    fold_metrics.append({
        "Classifier": clf_name,
        "Fold": fold+1,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "Confusion Matrix": cm.tolist()
    })

# averages for MLP
accs = [m["Accuracy"] for m in fold_metrics]
precs = [m["Precision"] for m in fold_metrics]
recs = [m["Recall"] for m in fold_metrics]
f1s = [m["F1-score"] for m in fold_metrics]
avg_results = {
    "Classifier": clf_name,
    "Fold": "Average",
    "Accuracy": np.mean(accs),
    "Accuracy_std": np.std(accs),
    "Precision": np.mean(precs),
    "Precision_std": np.std(precs),
    "Recall": np.mean(recs),
    "Recall_std": np.std(recs),
    "F1-score": np.mean(f1s),
    "F1-score_std": np.std(f1s),
    "Confusion Matrix": ""
}
fold_metrics.append(avg_results)
results.extend(fold_metrics)

# ---------- Transformer ----------
clf_name = "Transformer"
print(f"\n=== Running {clf_name} ===")
fold_metrics = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nProcessing Fold {fold+1}/{n_splits}")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    # Train transformer on training fold (we internally split again for val)
    model_tr = train_transformer_once(
        X_train_fold,
        y_train_fold,
        n_epochs=40,
        batch_size=32,
        lr=1e-3,
        d_model=64,
        nhead=4,
        num_layers=1
    )
    y_pred = predict_transformer(model_tr, X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
    rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Fold {fold+1} Test: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")
    print(f"Confusion Matrix for Fold {fold+1}:\n{cm}\n")

    fold_metrics.append({
        "Classifier": clf_name,
        "Fold": fold+1,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "Confusion Matrix": cm.tolist()
    })

# averages for Transformer
accs = [m["Accuracy"] for m in fold_metrics]
precs = [m["Precision"] for m in fold_metrics]
recs = [m["Recall"] for m in fold_metrics]
f1s = [m["F1-score"] for m in fold_metrics]
avg_results = {
    "Classifier": clf_name,
    "Fold": "Average",
    "Accuracy": np.mean(accs),
    "Accuracy_std": np.std(accs),
    "Precision": np.mean(precs),
    "Precision_std": np.std(precs),
    "Recall": np.mean(recs),
    "Recall_std": np.std(recs),
    "F1-score": np.mean(f1s),
    "F1-score_std": np.std(f1s),
    "Confusion Matrix": ""
}
fold_metrics.append(avg_results)
results.extend(fold_metrics)

# --- Save results ---
results_df = pd.DataFrame(results)
results_df.to_csv("mlp_transformer_cv_results_RFsubset.csv", index=False)
print("\nAll results saved to mlp_transformer_cv_results_RFsubset.csv")

Using 239 features common in at least 3 folds for Random Forest.
Using device: cuda

=== Running MLP ===

Processing Fold 1/5
Fold 1 Test: Acc=0.9474, Prec=0.9600, Rec=0.9730, F1=0.9664
Confusion Matrix for Fold 1:
[[18  3]
 [ 2 72]]


Processing Fold 2/5
Fold 2 Test: Acc=0.9053, Prec=0.9333, Rec=0.9459, F1=0.9396
Confusion Matrix for Fold 2:
[[16  5]
 [ 4 70]]


Processing Fold 3/5
Fold 3 Test: Acc=0.8947, Prec=0.9211, Rec=0.9459, F1=0.9333
Confusion Matrix for Fold 3:
[[15  6]
 [ 4 70]]


Processing Fold 4/5
Fold 4 Test: Acc=0.9149, Prec=0.9231, Rec=0.9730, F1=0.9474
Confusion Matrix for Fold 4:
[[14  6]
 [ 2 72]]


Processing Fold 5/5
Fold 5 Test: Acc=0.9255, Prec=0.9342, Rec=0.9726, F1=0.9530
Confusion Matrix for Fold 5:
[[16  5]
 [ 2 71]]


=== Running Transformer ===

Processing Fold 1/5
Fold 1 Test: Acc=0.8526, Prec=0.8750, Rec=0.9459, F1=0.9091
Confusion Matrix for Fold 1:
[[11 10]
 [ 4 70]]


Processing Fold 2/5
Fold 2 Test: Acc=0.8526, Prec=0.9054, Rec=0.9054, F1=0.9054
Confu

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=500)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.932±0.015, Precision = 0.945±0.017, Recall = 0.970±0.010, F1 Score = 0.957±0.009

Running classifier: Transformer
Transformer: Accuracy = 0.863±0.030, Precision = 0.904±0.015, Recall = 0.922±0.031, F1 Score = 0.913±0.020

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=600)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.915±0.018, Precision = 0.930±0.016, Recall = 0.965±0.022, F1 Score = 0.947±0.011

Running classifier: Transformer
Transformer: Accuracy = 0.873±0.006, Precision = 0.886±0.012, Recall = 0.962±0.016, F1 Score = 0.922±0.004

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=700)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.913±0.017, Precision = 0.934±0.012, Recall = 0.957±0.032, F1 Score = 0.945±0.012

Running classifier: Transformer
Transformer: Accuracy = 0.877±0.019, Precision = 0.903±0.015, Recall = 0.946±0.043, F1 Score = 0.923±0.015

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=800)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.922±0.025, Precision = 0.937±0.023, Recall = 0.965±0.011, F1 Score = 0.951±0.015

Running classifier: Transformer
Transformer: Accuracy = 0.882±0.034, Precision = 0.905±0.023, Recall = 0.949±0.037, F1 Score = 0.926±0.021

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=900)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.920±0.011, Precision = 0.937±0.014, Recall = 0.962±0.023, F1 Score = 0.949±0.007

Running classifier: Transformer
Transformer: Accuracy = 0.858±0.030, Precision = 0.896±0.018, Recall = 0.927±0.044, F1 Score = 0.910±0.021

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=1000)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.924±0.022, Precision = 0.933±0.023, Recall = 0.973±0.008, F1 Score = 0.952±0.013

Running classifier: Transformer
Transformer: Accuracy = 0.867±0.029, Precision = 0.898±0.028, Recall = 0.938±0.033, F1 Score = 0.917±0.018

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. ANOVA feature selection (top 500) ----------
selector = SelectKBest(score_func=f_classif, k=400)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)

# ---------- 3. Helper: Torch dataset & training for transformer ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treats each feature as a 'token'.
    - Input: (batch, n_features)
    - Linear projection to d_model
    - Positional embedding over feature positions
    - TransformerEncoder
    - CLS token + mean-pooling -> classification head
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings for each feature index
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional embedding

        # prepend CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # simple validation loss (optional early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    preds = []
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define the two classifiers ----------
classifiers = {
    "MLP": "MLP",             # placeholder key; we’ll special‑case in the loop
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation Setup ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            # Simple MLP classifier
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size='auto',
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            # Fit on train; we’re not explicitly using val here (could tune hyperparams)
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            # Train transformer model
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,       # adjust for more training
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # --- Average and std for each metric ---
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to mlp_transformer_cv_results.csv)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [  144   283   284   297   321   365   565   732   819   839   878  1077
  1100  1188  1255  1256  1257  1259  1338  1399  1425  1430  1459  1461
  1619  1655  1743  1765  1766  1891  1993  1994  2003  2031  2032  2224
  2225  2231  2236  2409  2476  2690  2691  2693  2698  2699  2733  2889
  2903  2942  2944  3039  3048  3049  3054  3076  3079  3206  3273  3284
  3288  3425  3551  3895  3994  4026  4036  4050  4051  4052  4135  4205
  4922  4933  5009  5033  5066  5242  5299  5371  5418  5439  5607  5608
  5609  5610  5648  5755  6030  6049  6051  6118  6125  6147  6323  6324
  6825  6852  6855  6856  6860  7207  7218  7357  7403  7420  7421  7440
  7602  7650  7857  7858  7967  7995  8000  8025  8107  8276  9029  9200
  9305  9485  9493  9495  9621  9625  9666  9755  9759  9969 10262 10315
 10458 10464 10502 10565 10608 10668 10690 10958 10959 10962 10963 10964
 10965


Running classifier: MLP
MLP: Accuracy = 0.922±0.015, Precision = 0.933±0.017, Recall = 0.970±0.013, F1 Score = 0.951±0.009

Running classifier: Transformer
Transformer: Accuracy = 0.898±0.021, Precision = 0.912±0.033, Recall = 0.965±0.025, F1 Score = 0.937±0.012

Done. (Optionally saved to mlp_transformer_cv_results.csv)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 400  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=30,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 400 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C',

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 500  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 500 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C',

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 600  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 600 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C',

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 700  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 700 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C',

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 800  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 800 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C',

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 900  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 900 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C',

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import ttest_ind

# ---------- 1. Load your data ----------
SKCM_transposed = pd.read_csv('SKCM_transposed.csv')  # CHANGE this filename

X = SKCM_transposed.drop(columns=['sample', 'index', 'TumorType'], errors='ignore')
y = (SKCM_transposed['TumorType'] == 'Metastatic').astype(int)

# ---------- 2. T‑test feature selection (univariate) ----------
k = 1000  # number of features to keep

# Separate groups: 0 vs 1
X0 = X[y == 0]
X1 = X[y == 1]

t_scores = []
for col in X.columns:
    # Welch’s t-test (equal_var=False is usually safer for bio data)
    t_stat, p_val = ttest_ind(X0[col].values, X1[col].values, equal_var=False, nan_policy='omit')
    t_scores.append(abs(t_stat))

t_scores = np.array(t_scores)
# indices of top-k features by |t-stat|
top_indices = np.argsort(t_scores)[::-1][:k]
selected_feature_names = X.columns[top_indices]

X_selected = X.iloc[:, top_indices]
X_selected_df = X_selected.copy()

print(f"\nSelected top {k} features by |t-statistic|:")
print(list(selected_feature_names))

# ---------- 3. Helper: Torch dataset & Transformer model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleTransformerClassifier(nn.Module):
    """
    Treat each feature as a token.
    Input: (batch, n_features)
    """
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Project scalar feature to d_model
        self.input_proj = nn.Linear(1, d_model)

        # Positional embeddings
        self.pos_emb = nn.Parameter(torch.randn(1, n_features, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        bsz, n_feats = x.shape
        x = x.view(bsz, n_feats, 1)               # (b, f, 1)
        x = self.input_proj(x)                    # (b, f, d_model)
        x = x + self.pos_emb[:, :n_feats, :]      # add positional emb

        # CLS token
        cls = self.cls_token.expand(bsz, -1, -1)  # (b, 1, d_model)
        x = torch.cat([cls, x], dim=1)            # (b, 1+f, d_model)

        x = self.encoder(x)                       # (b, 1+f, d_model)
        cls_out = x[:, 0, :]                      # (b, d_model)

        logits = self.classifier(cls_out).squeeze(-1)  # (b,)
        return logits


def train_transformer_model(X_train, y_train, X_val, y_val,
                            n_epochs=30, batch_size=32, lr=1e-3,
                            d_model=64, nhead=4, num_layers=2):
    n_features = X_train.shape[1]

    model = SimpleTransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=4 * d_model,
        dropout=0.1
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_transformer(model, X_test):
    model.eval()
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits)
        preds = (probs.cpu().numpy() >= 0.5).astype(int)
    return preds

# ---------- 4. Define classifiers (MLP + Transformer) ----------
classifiers = {
    "MLP": "MLP",
    "Transformer": "TRANSFORMER",
}

# ---------- 5. Cross‑Validation ----------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []

for clf_name, clf_type in classifiers.items():
    print(f"\nRunning classifier: {clf_name}")
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected_df, y)):
        X_train, X_test = X_selected_df.iloc[train_idx], X_selected_df.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # inner train/val split
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
        )

        if clf_type == "MLP":
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                random_state=42
            )
            mlp.fit(X_train_fold, y_train_fold)
            y_pred = mlp.predict(X_test)

        elif clf_type == "TRANSFORMER":
            transformer = train_transformer_model(
                X_train_fold, y_train_fold,
                X_val_fold, y_val_fold,
                n_epochs=100,
                batch_size=32,
                lr=1e-3,
                d_model=64,
                nhead=4,
                num_layers=2
            )
            y_pred = predict_transformer(transformer, X_test)

        else:
            raise ValueError(f"Unknown classifier type: {clf_type}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
        rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

        fold_metrics.append({
            "Classifier": clf_name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
        })

    # averages
    accs = [m["Accuracy"] for m in fold_metrics]
    precs = [m["Precision"] for m in fold_metrics]
    recs = [m["Recall"] for m in fold_metrics]
    f1s = [m["F1-score"] for m in fold_metrics]

    avg_results = {
        "Classifier": clf_name,
        "Fold": "Average",
        "Accuracy": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "Precision": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall": np.mean(recs),
        "Recall_std": np.std(recs),
        "F1-score": np.mean(f1s),
        "F1-score_std": np.std(f1s),
    }
    results.append(avg_results)

    print(
        f"{clf_name}: "
        f"Accuracy = {avg_results['Accuracy']:.3f}±{avg_results['Accuracy_std']:.3f}, "
        f"Precision = {avg_results['Precision']:.3f}±{avg_results['Precision_std']:.3f}, "
        f"Recall = {avg_results['Recall']:.3f}±{avg_results['Recall_std']:.3f}, "
        f"F1 Score = {avg_results['F1-score']:.3f}±{avg_results['F1-score_std']:.3f}"
    )

results_df = pd.DataFrame(results)
# results_df.to_csv("ttest_mlp_transformer_cv_results.csv", index=False)
print("\nDone. (Optionally saved to ttest_mlp_transformer_cv_results.csv)")


Selected top 1000 features by |t-statistic|:
['SNORD114-3', 'OR2T29', 'SNORD116-24', 'SNORD91B', 'SNORD47', 'KRTAP19-2', 'SNORD116-25', 'SNORD116-23', 'SNORD121A', 'HSFX1', 'SNORD96A', 'SNORD116-13', 'SNORD116-12', 'SNORA58', 'OR5H14', 'TTTY21', 'SNORD114-2', 'SNAR-A13', 'SNORD105', 'SNORD116-10', 'SNORD60', 'PRAMEF3', 'CDY1', 'SNORD16', 'TXNDC8', '?|728045', 'SNORD38A', 'SNORD116-29', 'LOC254312', 'SNORD36A', 'PRR20A', 'SNORD115-32', 'SNORD31', 'SNORD115-10', 'SNORD114-6', 'SNORD43', 'SNORD114-5', 'SNORD111', 'SNORD110', 'SNORD117', 'SNORD50A', 'DUX4', 'OR5B17', 'SNORD115-33', 'SNORD115-1', '?|441362', 'SNORD18B', 'SNORD86', 'SNORD116-16', 'SNORD115-30', 'SNAR-H', 'SCARNA23', 'SNORD58C', 'SNORD58A', 'SNORD116-14', 'DEFB105A', 'SNORD116-15', 'SNORD115-31', 'SNORD87', 'SNORD113-7', 'DEFB106A', 'SNORD115-20', 'SNORD115-22', 'SNORD115-25', 'DEFB130', 'SERPINA13', 'KRTAP25-1', 'SNORD114-8', 'TMEM225', 'SNORD45C', 'SNORD111B', 'RNU5E', 'SNORD18A', 'SPINT4', 'SNORD104', 'TTTY3B', 'SNORD88C'

In [ ]:
import json
import pandas as pd
import ast

# Example instruction template (you can later generate multiple variants)
INSTRUCTION = "Explain the model's prediction for this tumor in 2–3 sentences."

def clean_features(features):
    """
    Ensure all top_feature dictionaries have the required keys
    and are JSON-serializable.
    """
    cleaned = []

    for f in features:
        cleaned.append({
            "gene names": f.get("feature"),
            "gene expression value": float(f.get("raw_value")),
            "cohort_mean": float(f.get("cohort_mean")),
            "cohort_percentile": float(f.get("cohort_percentile")),
            "shap value": float(f.get("shap")),
            "direction": f.get("direction"),
            "interpretation": f.get("interpretation")
        })

    return cleaned


def build_training_example(row):
    """
    Convert one dataframe row into instruction-tuning JSON format.
    """

    # Parse top_features_json safely
    if isinstance(row["top_features_json"], str):
        top_features = ast.literal_eval(row["top_features_json"])
    else:
        top_features = row["top_features_json"]

    top_features = clean_features(top_features)

    record = {
        "instruction": INSTRUCTION,
        "input": {
            "true_label": row["true_label"],
            "model_prediction": row["model_prediction"],
            "model_probability_pos_class": round(float(row["model_probability_pos_class"]), 6),
            "top_features": top_features
        },

        # Optional placeholder output if you don't have gold explanations yet:
        "output": ""
    }

    return record


def dataframe_to_jsonl(df, output_path="training_data.jsonl"):
    """
    Convert whole DataFrame to JSONL format.
    """

    with open(output_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            example = build_training_example(row)
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

    print(f"✅ Saved training data to: {output_path}")


df = pd.read_csv("bundles_summary.csv")
dataframe_to_jsonl(df, "tumor_shap_training.jsonl")


✅ Saved training data to: tumor_shap_training.jsonl


In [ ]:
!nvidia-smi  # just to confirm you have a GPU

!pip install -q -U "transformers" "accelerate" "torch" pandas

Wed Nov 26 20:47:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
from transformers import pipeline

model_name = "microsoft/phi-2"

# Create a text-generation pipeline using Phi-2
pipe = pipeline(
    "text-generation",
    model=model_name,
    device_map="auto",         # puts model on GPU if available
    trust_remote_code=True,    # required for Phi-2 in transformers
)

# Quick test (optional)
test = pipe("Instruct: Explain what SHAP values mean in a model.\nOutput:",
            max_new_tokens=80,
            do_sample=False)[0]["generated_text"]
print(test)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruct: Explain what SHAP values mean in a model.
Output: SHAP values are a way of measuring how much each feature contributes to the prediction of a target variable in a model. They are based on the idea of perturbing the model by changing the value of one feature at a time and observing how the prediction changes. The SHAP values show the direction and magnitude of the effect of each feature on the prediction, relative to the baseline value of the feature


In [ ]:
import ast
import pandas as pd

INSTRUCTION = "Explain the model's prediction for this tumor in 2–3 sentences."

def parse_top_features(row):
    """Parse the top_features_json column into a Python list of dicts."""
    tf = row["top_features_json"]
    if isinstance(tf, str):
        features = ast.literal_eval(tf)
    else:
        features = tf
    return features

def build_prompt(row):
    """
    Build a prompt for Phi-2 using the row information.
    """
    true_label = row["true_label"]
    model_pred = row["model_prediction"]
    prob = float(row["model_probability_pos_class"])
    features = parse_top_features(row)

    feature_lines = []
    for f in features:
        feature_lines.append(
            f"- {f['feature']}: raw={f['raw_value']}, "
            f"mean={f['cohort_mean']}, "
            f"percentile={f['cohort_percentile']}, "
            f"SHAP={f['shap']} ({f['direction']}), "
            f"interpretation={f['interpretation']}"
        )
    feature_block = "\n".join(feature_lines)

    # Phi-2 likes simple "Instruct: ... Output:" style prompts
    prompt = f"""Instruct: You are helping explain a machine learning model that predicts whether a tumor is metastatic or primary based on gene expression and SHAP values.

True label: {true_label}
Model prediction: {model_pred}
Model metastatic probability: {prob:.3f}

Top contributing genes for this sample:
{feature_block}

Task: {INSTRUCTION}

Write the explanation for a clinician.
- Mention which genes increase metastatic probability and which decrease it.
- Refer to expression levels qualitatively using the percentiles (for example, "very low", "high").
- End by restating the final metastatic probability.
Output:"""

    return prompt

def generate_explanation_for_row(row, max_new_tokens=200):
    """
    Use Phi-2 to generate a natural-language explanation for one row.
    """
    prompt = build_prompt(row)

    out = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,      # deterministic, more consistent
        pad_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"]

    # The model will return prompt + completion. Strip the prompt prefix.
    # This is simple: take everything after the last occurrence of "Output:".
    if "Output:" in out:
        explanation = out.split("Output:", 1)[1].strip()
    else:
        explanation = out.strip()

    return explanation

In [ ]:
# Load your original CSV
df = pd.read_csv("bundles_summary.csv")

# Generate explanations row by row
explanations = []
for idx, row in df.iterrows():
    print(f"Generating explanation for row {idx}...")
    try:
        text = generate_explanation_for_row(row)
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        text = ""
    explanations.append(text)

df["llm_explanation"] = explanations


Generating explanation for row 0...
Generating explanation for row 1...
Generating explanation for row 2...
Generating explanation for row 3...
Generating explanation for row 4...
Generating explanation for row 5...
Generating explanation for row 6...
Generating explanation for row 7...
Generating explanation for row 8...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Generating explanation for row 9...
Generating explanation for row 10...
Generating explanation for row 11...
Generating explanation for row 12...
Generating explanation for row 13...
Generating explanation for row 14...
Generating explanation for row 15...
Generating explanation for row 16...
Generating explanation for row 17...
Generating explanation for row 18...
Generating explanation for row 19...
Generating explanation for row 20...
Generating explanation for row 21...
Generating explanation for row 22...
Generating explanation for row 23...
Generating explanation for row 24...
Generating explanation for row 25...
Generating explanation for row 26...
Generating explanation for row 27...
Generating explanation for row 28...
Generating explanation for row 29...
Generating explanation for row 30...
Generating explanation for row 31...
Generating explanation for row 32...
Generating explanation for row 33...
Generating explanation for row 34...
Generating explanation for row 35...
Ge

In [ ]:
df[["true_label", "model_prediction", "model_probability_pos_class", "llm_explanation"]].head()

,true_label,model_prediction,model_probability_pos_class,llm_explanation
0,Metastatic,Metastatic,0.941565,The model predicts that this tumor is metastat...
1,Metastatic,Metastatic,0.974216,The model predicts that this tumor is metastat...
2,Primary,Metastatic,0.955499,The model predicts that this tumor is metastat...
3,Metastatic,Metastatic,0.979447,The model predicts that this tumor is metastat...
4,Primary,Metastatic,0.941565,The model predicts that this tumor is metastat...


In [ ]:
import json

INSTRUCTION = "Explain the model's prediction for this tumor in 2–3 sentences."

def clean_features(features):
    cleaned = []
    for f in features:
        cleaned.append({
            "feature": f.get("feature"),
            "raw_value": float(f.get("raw_value")),
            "cohort_mean": float(f.get("cohort_mean")),
            "cohort_percentile": float(f.get("cohort_percentile")),
            "shap": float(f.get("shap")),
            "direction": f.get("direction"),
            "interpretation": f.get("interpretation"),
        })
    return cleaned

def build_training_example(row):
    features = parse_top_features(row)
    top_features = clean_features(features)

    record = {
        "instruction": INSTRUCTION,
        "input": {
            "true_label": row["true_label"],
            "model_prediction": row["model_prediction"],
            "model_probability_pos_class": round(float(row["model_probability_pos_class"]), 6),
            "top_features": top_features,
        },
        "output": row.get("llm_explanation", ""),
    }
    return record

def dataframe_to_jsonl(df, output_path="tumor_shap_training_with_outputs.jsonl"):
    with open(output_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            example = build_training_example(row)
            f.write(json.dumps(example, ensure_ascii=False) + "\n")
    print(f"✅ Saved training data to: {output_path}")

# Write JSONL
dataframe_to_jsonl(df, "tumor_shap_training_with_outputs.jsonl")

✅ Saved training data to: tumor_shap_training_with_outputs.jsonl


In [ ]:
import json
import pandas as pd

jsonl_path = "tumor_shap_training_with_outputs.jsonl"

# Load JSONL
rows = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

# Convert to DataFrame
df = pd.DataFrame(rows)

# Flatten the "input" column
input_df = pd.json_normalize(df["input"])
df = pd.concat([df.drop(columns=["input"]), input_df], axis=1)

# Convert complex lists into strings (CSV-safe)
df["top_features"] = df["top_features"].apply(json.dumps)

# Save to CSV
csv_path = "tumor_shap_training_with_outputs.csv"
df.to_csv(csv_path, index=False)

print("✅ CSV file saved as:", csv_path)

✅ CSV file saved as: tumor_shap_training_with_outputs.csv


In [ ]:
# Installs Unsloth, Xformers (for speed), and TRL (for training)
%%capture
!pip install unsloth
# Also get the latest nightly version for Llama 3.2 support
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None = auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False for 1B model but True is safer.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.11.3 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [ ]:
from datasets import load_dataset

# 1. Load your specific file
# Make sure 'train.jsonl' is uploaded to the Colab file area
dataset = load_dataset("json", data_files="train.jsonl", split="train")

# 2. Define how to combine Prompt + Completion
# We add an EOS (End of Sentence) token so the model knows when to stop writing.
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    completions = examples["completion"]
    texts = []
    for prompt, completion in zip(prompts, completions):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = prompt + "\n" + completion + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 3. Apply the formatting
dataset = dataset.map(formatting_prompts_func, batched = True,)

# Optional: See what the model sees
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/473 [00:00<?, ? examples/s]

Use ONLY the information below. Return EXACTLY one line:
1) Justification: one sentence stating which top features (name, percentile, SHAP) drove the model's prediction and why. Include the model probability.

Model prediction: Metastatic (probability = 0.941565)
Top drivers:
- C7: raw=5.6681, percentile=37.6%, SHAP=-1.657138 (low → decreases metastatic probability)
- MMP3: raw=0.9006, percentile=28.8%, SHAP=-0.891095 (low → decreases metastatic probability)
- EBF2: raw=0.0, percentile=8.0%, SHAP=0.785215 (low → increases metastatic probability)
- CDK14: raw=10.9263, percentile=99.2%, SHAP=-0.772393 (high → decreases metastatic probability)
- STMN2: raw=6.8125, percentile=94.3%, SHAP=0.644947 (high → increases metastatic probability)
Patient context: none provided

Return exactly:
Justification: ...
Justification: C7 (low -> decreases metastatic probability), MMP3 (low -> decreases metastatic probability), and EBF2 (low -> increases metastatic probability) together explain the model pr

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Adjust this based on how well it learns. Start with 100.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/473 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 473 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ananda15-2404 (ananda15-2404-daffodil-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.599000
2,2.609600
3,2.563300
4,2.447000
5,2.308500
6,2.133600
7,1.950800
8,1.801000
9,1.627000
10,1.423400


TrainOutput(global_step=100, training_loss=0.7887519484758377, metrics={'train_runtime': 372.6099, 'train_samples_per_second': 2.147, 'train_steps_per_second': 0.268, 'total_flos': 1376004154638336.0, 'train_loss': 0.7887519484758377, 'epoch': 1.6751054852320675})

In [ ]:
# 1. Put model in inference mode (makes it faster)
FastLanguageModel.for_inference(model)

# 2. Take a sample prompt (Copy one from your file or write a new one)
# MAKE SURE this matches the format of your "prompt" column exactly.
test_prompt = """Use ONLY the information below. Return EXACTLY one line:
1) Justification: one sentence stating which top features (name, percentile, SHAP) drove the model's prediction and why. Include the model probability.

Model prediction: Metastatic (probability = 0.914216)
Top drivers:
- C7: raw=9.4089, percentile=78.0%, SHAP=-1.037046 (high → increase metastatic probability)
- STMN2: raw=0.7259, percentile=18.8%, SHAP=-0.902501 (low → increase metastatic probability)
- DNAH17: raw=2.9144, percentile=4.7%, SHAP=-0.767513 (low → decreases metastatic probability)
- TNFSF11: raw=4.1167, percentile=71.5%, SHAP=-0.745511 (high → decreases metastatic probability)
- CDK14: raw=4.4545, percentile=4.7%, SHAP=0.717634 (low → increases metastatic probability)
Patient context: none provided

Return exactly:
Justification: ..."""

# 3. Tokenize the input
inputs = tokenizer(
    [test_prompt],
    return_tensors = "pt"
).to("cuda")

# 4. Generate the output
outputs = model.generate(
    **inputs,
    max_new_tokens = 128, # How long the answer can be
    use_cache = True
)

# 5. Decode the result
# We skip the prompt part to only see the new answer
print(tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0])

C7 (high -> increases metastatic probability), STMN2 (low -> increases metastatic probability), and DNAH17 (low -> decreases metastatic probability) together explain the model predicting Metastatic (P=0.914).


In [ ]:
model.save_pretrained("lora_model") # Local saving
# tokenizer.save_pretrained("lora_model")

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import re

# 1. Load and Split the data (90% Train, 10% Test)
dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset["test"]

print(f"Training samples: {len(dataset['train'])}")
print(f"Testing samples: {len(test_dataset)}")

# 2. Switch to Inference Mode
FastLanguageModel.for_inference(model)

# 3. Run Generation on the Test Set
references = [] # The "Gold" answers
predictions = [] # The "Model" answers

print("Generating answers for test set...")
for i in tqdm(range(len(test_dataset))):
    # Get the prompt and the true answer
    prompt = test_dataset[i]["prompt"]
    true_completion = test_dataset[i]["completion"]

    # Prepare input
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and clean up
    generated_text = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

    # Store them
    predictions.append(generated_text)
    references.append(true_completion)

print("Generation complete!")

Training samples: 425
Testing samples: 48
Generating answers for test set...


100%|██████████| 48/48 [02:32<00:00,  3.18s/it]

Generation complete!


In [ ]:
def extract_features(text):
    # This looks for words like C7, STMN2, CDK14, etc.
    # Assuming features are uppercase alphanumeric words length 2+
    return set(re.findall(r'\b[A-Z][A-Z0-9]+\b', text))

correct_features_count = 0
total_features_count = 0

print("\n--- Evaluation Report ---")
for i in range(len(predictions)):
    # Extract features from Gold and Model output
    gold_features = extract_features(references[i])
    pred_features = extract_features(predictions[i])

    # Calculate intersection (how many matched?)
    matches = gold_features.intersection(pred_features)

    # Add to totals
    correct_features_count += len(matches)
    total_features_count += len(gold_features)

    # Optional: Print first 3 examples to see with your eyes
    if i < 3:
        print(f"\nExample {i+1}:")
        print(f"Gold Features: {gold_features}")
        print(f"Pred Features: {pred_features}")
        print(f"Matches: {len(matches)}/{len(gold_features)}")

# Final Score
accuracy = (correct_features_count / total_features_count) * 100
print(f"\n=============================================")
print(f"Feature Retrieval Accuracy: {accuracy:.2f}%")


--- Evaluation Report ---

Example 1:
Gold Features: {'MMP3', 'CDK14', 'C7'}
Pred Features: {'MMP3', 'CDK14', 'C7'}
Matches: 3/3

Example 2:
Gold Features: {'DPYD', 'C7', 'TNFSF11'}
Pred Features: {'DPYD', 'C7', 'TNFSF11'}
Matches: 3/3

Example 3:
Gold Features: {'DMKN', 'C7', 'S100A7'}
Pred Features: {'DMKN', 'C7', 'S100A7'}
Matches: 3/3

Feature Retrieval Accuracy: 100.00%


In [ ]:
# Step A: Uninstall any potentially conflicting packages
!pip uninstall -y sentence-transformers gcsfs bitsandbytes

# Step B: Install the other required libraries
!pip install -q \
    "transformers==4.40.2" \
    "datasets==2.18.0" \
    "accelerate==0.30.1" \
    "peft==0.10.0" \
    "trl==0.8.6"

# Step C: Clone the bitsandbytes repository
!git clone https://github.com/TimDettmers/bitsandbytes.git

# Step D: Compile and install bitsandbytes from source using the correct method
%cd bitsandbytes
!make cuda11x

# This is the corrected command. `pip install .` is the modern, robust way
# to install a package from its source directory. It handles metadata correctly.
!pip install .
%cd ..

print("\n\n✅ bitsandbytes has been successfully compiled and installed from source.")
print("✅ This should be the final fix. Please proceed with the rest of the notebook.")

Found existing installation: sentence-transformers 5.1.2
Uninstalling sentence-transformers-5.1.2:
  Successfully uninstalled sentence-transformers-5.1.2
Found existing installation: gcsfs 2025.3.0
Uninstalling gcsfs-2025.3.0:
  Successfully uninstalled gcsfs-2025.3.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.0 MB/s eta 0:00:00
ERROR: pip's

In [ ]:
import datasets

# Load the dataset from the 'train.jsonl' file you uploaded.
dataset = datasets.load_dataset('json', data_files='train.jsonl', split='train')

# This function formats the prompt and completion into a single text field
# that the model will learn from. The format is crucial for instruction-tuned models.
def format_instruction(sample):
    full_completion = (
        f"{sample['completion']}\n"
    )
    return {
        "text": f"<s><|user|>\n{sample['prompt']}<|end|>\n<|assistant|>\n{full_completion}<|end|>"
    }

# Apply this formatting to every sample in our dataset.
formatted_dataset = dataset.map(format_instruction)

# Print the first formatted sample to verify it's correct.
print("Formatted dataset sample for the model:")
print(formatted_dataset[0]['text'])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/473 [00:00<?, ? examples/s]

Formatted dataset sample for the model:
<s><|user|>
Use ONLY the information below. Return EXACTLY one line:
1) Justification: one sentence stating which top features (name, percentile, SHAP) drove the model's prediction and why. Include the model probability.

Model prediction: Metastatic (probability = 0.941565)
Top drivers:
- C7: raw=5.6681, percentile=37.6%, SHAP=-1.657138 (low → decreases metastatic probability)
- MMP3: raw=0.9006, percentile=28.8%, SHAP=-0.891095 (low → decreases metastatic probability)
- EBF2: raw=0.0, percentile=8.0%, SHAP=0.785215 (low → increases metastatic probability)
- CDK14: raw=10.9263, percentile=99.2%, SHAP=-0.772393 (high → decreases metastatic probability)
- STMN2: raw=6.8125, percentile=94.3%, SHAP=0.644947 (high → increases metastatic probability)
Patient context: none provided

Return exactly:
Justification: ...<|end|>
<|assistant|>
Justification: C7 (low -> decreases metastatic probability), MMP3 (low -> decreases metastatic probability), and EBF

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "microsoft/Phi-3-mini-4k-instruct"

# Configure 4-bit quantization to reduce memory usage.
# This configuration remains the same.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load the model with the specified quantization config.
# We have added `attn_implementation="eager"` for maximum stability.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
    attn_implementation="eager" # This is the "compatibility mode" setting.
)
model.config.use_cache = False # Recommended for fine-tuning

# Load the tokenizer for the Phi-3 model.
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model and tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: 
🚨 Forgot to compile the bitsandbytes library? 🚨
1. You're not using the package but checked-out the source code
2. You MUST compile from source

Attempted to use bitsandbytes native library functionality but it's not available.

This typically happens when:
1. bitsandbytes doesn't ship with a pre-compiled binary for your CUDA version
2. The library wasn't compiled properly during installation from source

To make bitsandbytes work, the compiled library version MUST exactly match the linked CUDA version.
If your CUDA version doesn't have a pre-compiled binary, you MUST compile from source.

You have two options:
1. COMPILE FROM SOURCE (required if no binary exists):
   https://huggingface.co/docs/bitsandbytes/main/en/installation#cuda-compile
2. Use BNB_CUDA_VERSION to specify a DIFFERENT CUDA version from the detected one, which is installed on your machine and matching an available pre-compiled version listed above

Original error: Configured CUDA binary not found at /u

In [ ]:
# This command installs the libraries needed to load, fine-tune, and serve the model.
# - transformers: The main library from Hugging Face to work with models and pipelines.
# - datasets: For loading and processing your .jsonl file easily.
# - accelerate: Optimizes PyTorch training on any infrastructure, including Colab GPUs.
# - bitsandbytes: Enables 4-bit quantization, a technique to load large models on smaller GPUs.
# - peft: Parameter-Efficient Fine-Tuning, which allows us to fine-tune the model without training all of its parameters.
# - trl: Transformer Reinforcement Learning library, contains the SFTTrainer we will use.
!pip install -q transformers datasets accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 30.8 MB/s eta 0:00:00


In [ ]:
import datasets

# Load the dataset from the 'train.jsonl' file you uploaded.
# The 'datasets' library automatically handles JSON Lines format.
dataset = datasets.load_dataset('json', data_files='train.jsonl', split='train')

# Let's look at the first example to see what we're working with.
print("Original dataset sample:")
print(dataset[0])

# The Phi-3 model expects a specific instruction format. We need to format
# our 'prompt' and 'completion' columns into this structure.
# The format is: <s><|user|>\n{prompt}<|end|>\n<|assistant|>\n{completion}<|end|>
def format_instruction(sample):
    # The 'completion' from your file only contains the justification.
    # The prompt expects three lines, so we construct the full expected completion.
    # We can hardcode the 'Next step' and 'Caveat' as they are static in your prompts.
    full_completion = (
        f"{sample['completion']}\n"
        "Next step: The patient's case should be reviewed by a multidisciplinary team.\n"
        "Caveat: This model is for research purposes only and is not a clinical recommendation."
    )
    return {
        "text": f"<s><|user|>\n{sample['prompt']}<|end|>\n<|assistant|>\n{full_completion}<|end|>"
    }

# Apply this formatting to every sample in our dataset.
formatted_dataset = dataset.map(format_instruction)

# Let's check the formatted version of the first sample.
print("\nFormatted dataset sample for the model:")
print(formatted_dataset[0]['text'])

Generating train split: 0 examples [00:00, ? examples/s]

Original dataset sample:
{'prompt': "Use ONLY the information below. Return EXACTLY three lines:\n1) Justification: one sentence stating which top features (name, percentile, SHAP) drove the model's prediction and why. Include the model probability.\n\nModel prediction: Metastatic (probability = 0.941565)\nTop drivers:\n- C7: raw=5.6681, percentile=37.6%, SHAP=-1.657138 (low → decreases metastatic probability)\n- MMP3: raw=0.9006, percentile=28.8%, SHAP=-0.891095 (low → decreases metastatic probability)\n- EBF2: raw=0.0, percentile=8.0%, SHAP=0.785215 (low → increases metastatic probability)\n- CDK14: raw=10.9263, percentile=99.2%, SHAP=-0.772393 (high → decreases metastatic probability)\n- STMN2: raw=6.8125, percentile=94.3%, SHAP=0.644947 (high → increases metastatic probability)\nPatient context: none provided\n\nReturn exactly:\nJustification: ...\nNext step: ...\nCaveat: ...", 'completion': 'Justification: C7 (low -> decreases metastatic probability), MMP3 (low -> decreases metast

Map:   0%|          | 0/473 [00:00<?, ? examples/s]


Formatted dataset sample for the model:
<s><|user|>
Use ONLY the information below. Return EXACTLY three lines:
1) Justification: one sentence stating which top features (name, percentile, SHAP) drove the model's prediction and why. Include the model probability.

Model prediction: Metastatic (probability = 0.941565)
Top drivers:
- C7: raw=5.6681, percentile=37.6%, SHAP=-1.657138 (low → decreases metastatic probability)
- MMP3: raw=0.9006, percentile=28.8%, SHAP=-0.891095 (low → decreases metastatic probability)
- EBF2: raw=0.0, percentile=8.0%, SHAP=0.785215 (low → increases metastatic probability)
- CDK14: raw=10.9263, percentile=99.2%, SHAP=-0.772393 (high → decreases metastatic probability)
- STMN2: raw=6.8125, percentile=94.3%, SHAP=0.644947 (high → increases metastatic probability)
Patient context: none provided

Return exactly:
Justification: ...
Next step: ...
Caveat: ...<|end|>
<|assistant|>
Justification: C7 (low -> decreases metastatic probability), MMP3 (low -> decreases m

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The identifier for the pre-trained Phi-3 model on Hugging Face.
model_id = "microsoft/Phi-3-mini-4k-instruct"

# Configure 4-bit quantization to reduce memory usage.
# This allows us to run a powerful model on a free Colab GPU.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load the model with the specified quantization config.
# `trust_remote_code=True` is required for Phi-3.
# `device_map="auto"` automatically places the model on the GPU.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)
model.config.use_cache = False # Recommended for fine-tuning

# Load the tokenizer for the Phi-3 model.
# The tokenizer converts text into a format the model understands.
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# Set a padding token. This is a common practice.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Pad on the right to prevent issues with generation.

print("Model and tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


In [ ]:
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

# Configure LoRA for efficient fine-tuning.
# This configuration remains the same.
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# Define the training arguments.
# This configuration also remains the same.
training_args = TrainingArguments(
    output_dir="./phi3-finetuned-results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    max_steps=-1,
    fp16=True,
)

# Create the trainer instance.
# We have removed the 'tokenizer' argument from the initialization.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    max_seq_length=1024,
    packing=False,
)

# We add the tokenizer to the .train() call here.
# This is the format expected by the version of the library you have installed.
print("Starting the fine-tuning process...")
trainer.train(tokenizer=tokenizer)
print("Fine-tuning complete!")

# Save the final fine-tuned model adapter.
trainer.save_model("./phi3-finetuned-model")
print("Fine-tuned model adapter saved.")

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'